# IASTAM Onboard Wildfire Detection: Production Model Training & Flight Pipeline
**6U CubeSat Edge Intelligence · Multi-Spectral Tiny U-Net Benchmark & Edge Packaging**

**Pipeline (all stages implemented and tested in this notebook):**

```
raw uint16 frame ──► Gate 1 (input validation, transient detection)
                 ──► Preprocessing (DN → reflectance / brightness temperature)
                 ──► Tiny U-Net (ONNX FP32 / INT8)
                 ──► Decision Layer (threshold, connected components, events)
                 ──► Gate 2 (output validation)
                 ──► ≤200-byte alert packet (CRC)
                 ──► Persistent priority queue + contact windows + retry
```

**Bands.** Primary configuration `pereira3c` (SWIR3) = Landsat-8 c7 (SWIR2), c6 (SWIR1), c2 (blue) — the validated 3-band spectral set (Pereira et al.).
`b10` (LWIR) and `pereira3c_b10` are evaluated as thermal/control configurations.

**Operational Highlights & Methodology:**
* **Non-Circular Zero-Shot Evaluation:** Cell 20 re-evaluates the trained model on human-annotated scenes to remove label algorithm bias.
* **Transient & Noise Scrubbing:** Gate 1 scrubs single-event radiation upsets (SEU) and hot pixels before inference.
* **Edge Quantization & Verification:** Full FP32 ONNX export and static INT8 quantization with bitwise consistency checks.
* **On-Target Raspberry Pi Bundle:** Produces benchmark bundles for CubeSat SBC validation (e.g. Raspberry Pi CM4 / COTS AI accelerator).

**Execution Sequence:**
Cells 1 through 23 run sequentially to reproduce dataset preparation, model training, threshold calibration, INT8 export, and telemetry packet generation.


In [1]:
# ================================================================
# CELL 1 — ENVIRONMENT / PATHS / CONFIGURATION  (V7, multiband, standalone)
# ================================================================
# V7.1 roles:
#   1) REFERENCE_CONFIG = Pereira's validated 3-band C7+C6+C2 benchmark.
#   2) FLIGHT_CONFIG    = current SWIR3 payload proxy (same C7+C6+C2 spectral triplet).
#   3) CONTROL_CONFIG   = B10-only thermal control; not the current payload choice.
# The actual flight sensor must later replace the Landsat proxy with its own raw-DN
# contract, spectral response calibration, noise model and GSD/geometry metadata.

!pip -q install rasterio tqdm scipy pandas scikit-learn onnx onnxruntime onnxsim onnxscript gdown

import os
import sys
import gc
import json
import math
import random
import re
import shutil
import struct
import tempfile
import time
import zipfile
import zlib
import sqlite3
import hashlib
import platform
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import transform as rio_transform
from scipy import ndimage
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------------- paths ----------------
WORK_ROOT = Path('/kaggle/working')
SCRATCH_ROOT = Path('/kaggle/tmp/fireedge')
OUTPUT_DIR = WORK_ROOT / 'v7_outputs'
DATA_ROOT = SCRATCH_ROOT / 'africa'
CACHE_DIR = SCRATCH_ROOT / 'cache_v7'
for p in (WORK_ROOT, SCRATCH_ROOT, OUTPUT_DIR, DATA_ROOT, CACHE_DIR):
    p.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------- bands ----------------
# Pereira 10-band patch = Landsat-8 c1..c7, c9, c10, c11 -> rasterio band numbers 1..10.
#   c2  = blue            (rasterio band 2)
#   c6  = SWIR1 1.6 um    (rasterio band 6)
#   c7  = SWIR2 2.2 um    (rasterio band 7)
#   b10 = TIRS-1 LWIR     (rasterio band 9)
CACHE_BANDS = [('c2', 2), ('c6', 6), ('c7', 7), ('b10', 9)]

BAND_CONFIGS = {
    'pereira3c': ['c7', 'c6', 'c2'],
    'b10': ['b10'],
    'pereira3c_b10': ['c7', 'c6', 'c2', 'b10'],
}
REFERENCE_CONFIG = 'pereira3c'
FLIGHT_CONFIG = 'pereira3c'
CONTROL_CONFIG = 'b10'
PRIMARY_CONFIG = FLIGHT_CONFIG
MODEL_IDS = {'pereira3c': 1, 'b10': 2, 'pereira3c_b10': 3}

RUN_CONFIGS = ['pereira3c', 'b10', 'pereira3c_b10']

# ---------------- training ----------------
BATCH_SIZE = 16 if device.type == 'cuda' else 4
NUM_WORKERS = 0
PIN_MEMORY = device.type == 'cuda'
EPOCHS = 20
PATIENCE = 12
POS_WEIGHT = 10.0
MODEL_BASE = 16

# Explicit fire-size sampling target. Unlike the old inverse-density rule, this
# guarantees that tiny 1–5 px fires are NOT allowed to fall to a minority share.
# Expected sampled shares (when all buckets exist):
#   0 px=15%, 1–5 px=45%, 6–25 px=25%, 26–100 px=10%, >100 px=5%.
SIZE_TARGET_SHARE = np.array([0.15, 0.45, 0.25, 0.10, 0.05], dtype=np.float64)

# ---------------- validation / mission reporting ----------------
# These remain MISSION TARGETS, not guaranteed constraints for the current dataset.
MIN_FRAME_RECALL = 0.95
MAX_FRAME_FAR = 0.10

# ---------------- current prototype packet / test geometry ----------------
# Current Pereira reference experiment is Landsat at 30 m pixels. The flight case will
# supply its actual GSD at runtime; never hard-code this 30 m value in flight firmware.
GSD_M_TEST = 30.0
LOCATION_UNCERTAINTY_UNKNOWN_M = 65535
HEARTBEAT_EVERY_N_FRAMES = 30
GATE1_SCRUB_DEFAULT = False

# ---------------- run switches ----------------
CACHE_THREADS = 3
FREE_PATCHES_AFTER_CACHE = True
ROBUSTNESS_N = 1000
VERIFY_N = 1500
PIPELINE_N = 1500
RUN_MANUAL_ZERO_SHOT = True

print('OUTPUT_DIR:', OUTPUT_DIR)
print('CACHE_DIR :', CACHE_DIR)
print('DEVICE    :', device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
print('Reference configuration:', REFERENCE_CONFIG, '->', BAND_CONFIGS[REFERENCE_CONFIG])
print('Current payload proxy:', FLIGHT_CONFIG, '->', BAND_CONFIGS[FLIGHT_CONFIG])
print('Thermal control:', CONTROL_CONFIG, '->', BAND_CONFIGS[CONTROL_CONFIG])
print('Primary configuration for this V7 run:', PRIMARY_CONFIG)
print('Configurations to train:', RUN_CONFIGS)
print('Size-target sampling:', SIZE_TARGET_SHARE.tolist())
!df -h /kaggle/working /kaggle/tmp 2>/dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 72.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 75.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.8/185.8 kB 8.5 MB/s eta 0:00:00
OUTPUT_DIR: /kaggle/working/v7_outputs
CACHE_DIR : /kaggle/tmp/fireedge/cache_v7
DEVICE    : cuda Tesla T4
Reference configuration: pereira3c -> ['c7', 'c6', 'c2']
Current payload proxy: pereira3c -> ['c7', 'c6', 'c2']
Thermal control: b10 -> ['b10']
Primary configuration for this V7 run: pereira3c
Configurations to train: ['pereira3c', 'b10', 'pereira3c_b10']
Size-target sampling: [0.15, 0.45, 0.25, 0.1, 0.05]
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  224K   20G   1% /kaggle/working
overlay         8.0T  7.0T  1.1T  87% /


In [2]:
# ================================================================
# CELL 2 — DOWNLOAD AFRICA.ZIP ONLY
# ================================================================

import gdown
from pathlib import Path

AFRICA_FILE_ID = "1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX"
AFRICA_ARCHIVE = DATA_ROOT / "Africa.zip"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

if AFRICA_ARCHIVE.exists():
    size_gb = AFRICA_ARCHIVE.stat().st_size / (1024 ** 3)
    if size_gb < 0.01:
        print("Existing Africa.zip looks invalid/suspicious; removing it.")
        AFRICA_ARCHIVE.unlink()
    else:
        print(f"Africa.zip already exists: {size_gb:.2f} GB")

if not AFRICA_ARCHIVE.exists():
    print("Downloading Africa.zip to /kaggle/tmp ...")

    gdown.download(
        id=AFRICA_FILE_ID,
        output=str(AFRICA_ARCHIVE),
        quiet=False,
    )

assert AFRICA_ARCHIVE.exists(), "Africa.zip was not downloaded."

archive_size_gb = AFRICA_ARCHIVE.stat().st_size / (1024 ** 3)

assert archive_size_gb > 0.01, (
    f"Africa.zip appears too small: {archive_size_gb:.4f} GB"
)

print()
print(f"✅ Africa.zip downloaded")
print(f"Path: {AFRICA_ARCHIVE}")
print(f"Size: {archive_size_gb:.2f} GB")

print("\nScratch storage:")
!du -sh /kaggle/tmp/fireedge 2>/dev/null || true

Downloading...
From (original): https://drive.google.com/uc?id=1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX
From (redirected): https://drive.google.com/uc?id=1Ng3JwsjJPApshk8lJdGcsNHI52NaEMDX&confirm=t&uuid=15a5486a-868d-44e0-9684-c90fdbb1c73c
To: /kaggle/tmp/fireedge/africa/Africa.zip
100%|██████████| 31.7G/31.7G [08:19<00:00, 63.5MB/s]


✅ Africa.zip downloaded
Path: /kaggle/tmp/fireedge/africa/Africa.zip
Size: 29.49 GB

Scratch storage:
30G	/kaggle/tmp/fireedge


In [3]:
# ================================================================
# CELL 3 — INSPECT THE AFRICA ARCHIVE STRUCTURE
# ================================================================

import re
import zipfile
from pathlib import Path

with zipfile.ZipFile(AFRICA_ARCHIVE, "r") as archive:
    members = archive.namelist()

print(f"Total members in Africa.zip: {len(members):,}")

# ----------------------------------------------------------------
# Identify ONLY real scene image archives:
#   z166052.zip
#   z166053.zip
# etc.
#
# Explicitly excludes:
#   z166052_masks.zip
#   z166052_masks_derivates.zip
#   anything else
# ----------------------------------------------------------------

image_scene_archives = []

for member in members:
    name = Path(member).name

    if re.fullmatch(r"z\d+\.zip", name, flags=re.IGNORECASE):
        image_scene_archives.append(member)

# ----------------------------------------------------------------
# Identify ONLY matching mask archives:
#   z166052_masks.zip
# ----------------------------------------------------------------

mask_scene_archives = []

for member in members:
    name = Path(member).name

    if re.fullmatch(r"z\d+_masks\.zip", name, flags=re.IGNORECASE):
        mask_scene_archives.append(member)

image_scene_archives = sorted(image_scene_archives)
mask_scene_archives = sorted(mask_scene_archives)

# ----------------------------------------------------------------
# Scene keys
# ----------------------------------------------------------------

def image_scene_key(member):
    """
    z166052.zip -> z166052
    """
    return Path(member).stem.lower()


def mask_scene_key(member):
    """
    z166052_masks.zip -> z166052
    """
    name = Path(member).name.lower()
    return re.sub(r"_masks\.zip$", "", name, flags=re.IGNORECASE)


image_scene_keys = {
    image_scene_key(m)
    for m in image_scene_archives
}

mask_scene_keys = {
    mask_scene_key(m)
    for m in mask_scene_archives
}

matched_scene_keys = sorted(image_scene_keys & mask_scene_keys)
image_only_scene_keys = sorted(image_scene_keys - mask_scene_keys)
mask_only_scene_keys = sorted(mask_scene_keys - image_scene_keys)

# ----------------------------------------------------------------
# Report
# ----------------------------------------------------------------

print()
print("========== AFRICA ARCHIVE STRUCTURE ==========")
print(f"Real image scene archives:       {len(image_scene_archives):,}")
print(f"Real mask scene archives:        {len(mask_scene_archives):,}")
print(f"Matched image+mask scenes:       {len(matched_scene_keys):,}")
print(f"Images without matching mask:    {len(image_only_scene_keys):,}")
print(f"Masks without matching image:    {len(mask_only_scene_keys):,}")

print()
print("Examples of IMAGE archives:")
for m in image_scene_archives[:10]:
    print("  ", m)

print()
print("Examples of MASK archives:")
for m in mask_scene_archives[:10]:
    print("  ", m)

# ----------------------------------------------------------------
# Confirm that derivative archives are NOT being treated as images
# ----------------------------------------------------------------

derivative_archives = [
    m for m in members
    if Path(m).name.lower().endswith("_masks_derivates.zip")
]

print()
print(f"Excluded *_masks_derivates.zip archives: {len(derivative_archives):,}")

assert image_scene_archives, "No real image scene archives found."
assert mask_scene_archives, "No real mask scene archives found."
assert matched_scene_keys, "No image/mask scene pairs found."

print()
print("✅ Archive structure looks valid.")
print("✅ Derivative mask archives are excluded.")
print("✅ Matched scenes identified.")

Total members in Africa.zip: 1,475

========== AFRICA ARCHIVE STRUCTURE ==========
Real image scene archives:       513
Real mask scene archives:        513
Matched image+mask scenes:       513
Images without matching mask:    0
Masks without matching image:    0

Examples of IMAGE archives:
   z152074.zip
   z153074.zip
   z153075.zip
   z154074.zip
   z154075.zip
   z156063.zip
   z158069.zip
   z158070.zip
   z158071.zip
   z158072.zip

Examples of MASK archives:
   z152074_masks.zip
   z153074_masks.zip
   z153075_masks.zip
   z154074_masks.zip
   z154075_masks.zip
   z156063_masks.zip
   z158069_masks.zip
   z158070_masks.zip
   z158071_masks.zip
   z158072_masks.zip

Excluded *_masks_derivates.zip archives: 449

✅ Archive structure looks valid.
✅ Derivative mask archives are excluded.
✅ Matched scenes identified.


In [4]:
# ================================================================
# CELL 4 — EXTRACT MATCHED PATCHES + ALGORITHMIC MASKS
# ================================================================

import shutil
import tempfile
import zipfile
import re
from pathlib import Path

from tqdm.auto import tqdm

# ----------------------------------------------------------------
# Output directories
# ----------------------------------------------------------------

PREPARED_ROOT = DATA_ROOT / "africa_prepared"
PATCH_DIR = PREPARED_ROOT / "patches"
ALGORITHM_MASK_DIR = PREPARED_ROOT / "masks" / "algorithmic"

# Clean any previous partial extraction from this session.
if PREPARED_ROOT.exists():
    print(f"Removing previous prepared data: {PREPARED_ROOT}")
    shutil.rmtree(PREPARED_ROOT)

PATCH_DIR.mkdir(parents=True, exist_ok=True)
ALGORITHM_MASK_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------
# Build lookup tables from CELL 4
# ----------------------------------------------------------------

image_member_by_scene = {
    image_scene_key(member): member
    for member in image_scene_archives
}

mask_member_by_scene = {
    mask_scene_key(member): member
    for member in mask_scene_archives
}

matched_scene_keys = sorted(
    set(image_member_by_scene) & set(mask_member_by_scene)
)

print(f"Scenes to extract: {len(matched_scene_keys):,}")

# ----------------------------------------------------------------
# Helpers
# ----------------------------------------------------------------

def canonical_patch_name(name):
    """
    Converts algorithm-specific mask names back to the base patch stem.

    Example:
        abc_p01_kumar-roy.tif -> abc_p01
        abc_p01_murphy.tif    -> abc_p01
        abc_p01_schroeder.tif -> abc_p01
        abc_p01_voting.tif    -> abc_p01
    """
    stem = Path(name).stem

    stem = re.sub(
        r"_(?:kumar[-_ ]?roy|kumarroy|murphy|schroeder)"
        r"(?=_p|$)",
        "",
        stem,
        flags=re.IGNORECASE,
    )

    stem = re.sub(
        r"_(?:voting|intersection|mask)"
        r"(?=_p|$)",
        "",
        stem,
        flags=re.IGNORECASE,
    )

    return stem


def detect_algorithm(name):
    low = Path(name).name.lower()

    if "kumar" in low and "roy" in low:
        return "kumar_roy"

    if "murphy" in low:
        return "murphy"

    if "schroeder" in low:
        return "schroeder"

    return None


def open_nested_zip(outer_archive, member):
    """
    Open one nested ZIP without extracting the entire outer archive.

    The temporary file is deleted automatically when closed.
    """

    temp_file = tempfile.TemporaryFile()

    with zipfile.ZipFile(outer_archive, "r") as outer:
        with outer.open(member) as src:
            shutil.copyfileobj(src, temp_file, length=16 * 1024 * 1024)

    temp_file.seek(0)

    nested = zipfile.ZipFile(temp_file, "r")

    return temp_file, nested


# ----------------------------------------------------------------
# 1. Extract ONLY image TIFFs from matched scene archives
# ----------------------------------------------------------------

image_names = []
image_stems = set()
duplicate_images = []

with zipfile.ZipFile(AFRICA_ARCHIVE, "r"):

    for scene in tqdm(
        matched_scene_keys,
        desc="Extracting matched image scenes"
    ):

        member = image_member_by_scene[scene]

        temp_file, nested = open_nested_zip(
            AFRICA_ARCHIVE,
            member
        )

        try:
            for info in nested.infolist():

                if info.is_dir():
                    continue

                if not info.filename.lower().endswith((".tif", ".tiff")):
                    continue

                name = Path(info.filename).name
                stem = Path(name).stem

                destination = PATCH_DIR / name

                # Protect against silent overwriting.
                if destination.exists():
                    duplicate_images.append(name)
                    continue

                with nested.open(info) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(
                        src,
                        dst,
                        length=16 * 1024 * 1024
                    )

                image_names.append(name)
                image_stems.add(stem)

        finally:
            nested.close()
            temp_file.close()


# ----------------------------------------------------------------
# 2. Extract ONLY the 3 algorithmic masks used for voting
# ----------------------------------------------------------------

saved_masks = 0
mask_names = []

with zipfile.ZipFile(AFRICA_ARCHIVE, "r"):

    for scene in tqdm(
        matched_scene_keys,
        desc="Extracting algorithmic masks"
    ):

        member = mask_member_by_scene[scene]

        temp_file, nested = open_nested_zip(
            AFRICA_ARCHIVE,
            member
        )

        try:
            for info in nested.infolist():

                if info.is_dir():
                    continue

                if not info.filename.lower().endswith((".tif", ".tiff")):
                    continue

                name = Path(info.filename).name

                algorithm = detect_algorithm(name)

                # Ignore intersection/voting/derivative products here.
                if algorithm is None:
                    continue

                target = canonical_patch_name(name)

                # Keep only masks corresponding to extracted images.
                if target not in image_stems:
                    continue

                destination = ALGORITHM_MASK_DIR / name

                if destination.exists():
                    continue

                with nested.open(info) as src, open(destination, "wb") as dst:
                    shutil.copyfileobj(
                        src,
                        dst,
                        length=16 * 1024 * 1024
                    )

                saved_masks += 1
                mask_names.append(name)

        finally:
            nested.close()
            temp_file.close()


# ----------------------------------------------------------------
# 3. Summary
# ----------------------------------------------------------------

patch_count = len(list(PATCH_DIR.glob("*.tif")))
patch_count += len(list(PATCH_DIR.glob("*.tiff")))

mask_count = len(list(ALGORITHM_MASK_DIR.glob("*.tif")))
mask_count += len(list(ALGORITHM_MASK_DIR.glob("*.tiff")))

print()
print("========== EXTRACTION COMPLETE ==========")
print(f"Matched scenes:             {len(matched_scene_keys):,}")
print(f"Image patches extracted:    {patch_count:,}")
print(f"Algorithm masks extracted: {mask_count:,}")

if duplicate_images:
    print()
    print(
        f"⚠️ Duplicate image filenames skipped: "
        f"{len(duplicate_images):,}"
    )
else:
    print("✅ No duplicate image filenames detected.")

print()
print("Prepared data:")
print(f"  patches: {PATCH_DIR}")
print(f"  masks:   {ALGORITHM_MASK_DIR}")

print()
print("Scratch usage:")
!du -sh /kaggle/tmp/fireedge 2>/dev/null || true

Scenes to extract: 513


Extracting matched image scenes:   0%|          | 0/513 [00:00<?, ?it/s]

Extracting algorithmic masks:   0%|          | 0/513 [00:00<?, ?it/s]


========== EXTRACTION COMPLETE ==========
Matched scenes:             513
Image patches extracted:    27,729
Algorithm masks extracted: 68,005
✅ No duplicate image filenames detected.

Prepared data:
  patches: /kaggle/tmp/fireedge/africa/africa_prepared/patches
  masks:   /kaggle/tmp/fireedge/africa/africa_prepared/masks/algorithmic

Scratch usage:
68G	/kaggle/tmp/fireedge


In [5]:
# ================================================================
# CELL 5 — BUILD THE OFFICIAL VOTING TARGET FOR EVERY IMAGE
# ================================================================

# The published ActiveFire voting target requires >=2 methods to agree.
# Therefore an image with zero/one detected algorithmic mask gets a zero
# voting target; it is a voting-negative, not a missing training sample.

VOTING_DIR = PREPARED_ROOT / 'masks' / 'voting'
VOTING_DIR.mkdir(parents=True, exist_ok=True)

image_files = sorted(list(PATCH_DIR.glob('*.tif')) + list(PATCH_DIR.glob('*.tiff')))

masks_by_image = defaultdict(dict)
for mask_file in ALGORITHM_MASK_DIR.glob('*.tif'):
    alg = detect_algorithm(mask_file.name)
    target = canonical_patch_name(mask_file.name)
    if alg is not None:
        masks_by_image[target][alg] = mask_file

records = []
coverage = []

for image_file in tqdm(image_files, desc='Building voting masks'):
    stem = image_file.stem
    sources = masks_by_image.get(stem, {})
    coverage.append({
        'stem': stem,
        'n_algorithm_masks': len(sources),
    })

    with rasterio.open(image_file) as img_src:
        profile = img_src.profile.copy()
        height, width = img_src.height, img_src.width

    arrays = []
    for alg in ('kumar_roy', 'murphy', 'schroeder'):
        path = sources.get(alg)
        if path is None:
            continue
        with rasterio.open(path) as mask_src:
            arr = (mask_src.read(1) > 0).astype(np.uint8)
            if arr.shape != (height, width):
                raise ValueError(
                    f'Shape mismatch for {stem}: {arr.shape} vs {(height, width)}'
                )
            arrays.append(arr)

    if arrays:
        vote = (
            np.sum(arrays, axis=0) >= 2
        ).astype(np.uint8)
    else:
        vote = np.zeros((height, width), dtype=np.uint8)

    for k in ('blockxsize', 'blockysize', 'tiled'):
        profile.pop(k, None)
    profile.update(
        driver='GTiff',
        count=1,
        dtype='uint8',
        nodata=0,
        compress='lzw',
    )

    output = VOTING_DIR / f'{stem}_voting.tif'
    with rasterio.open(output, 'w', **profile) as dst:
        dst.write(vote, 1)

    records.append({
        'stem': stem,
        'image_path': str(image_file),
        'mask_path': str(output),
        'fire_pixels': int(vote.sum()),
        'n_algorithm_masks': len(arrays),
        'sample_type': 'voting_fire' if vote.any() else 'voting_zero',
    })

coverage_df = pd.DataFrame(coverage)
africa_df = pd.DataFrame(records)

assert len(africa_df) == len(image_files)

africa_df.to_csv(
    PREPARED_ROOT / 'africa_voting_inventory.csv',
    index=False,
)

print('Total African patches:', len(africa_df))
print('\nLabel distribution:')
display(africa_df['sample_type'].value_counts())
print('\nAlgorithm-mask coverage:')
display(coverage_df['n_algorithm_masks'].value_counts().sort_index())
print('\nFire-pixel statistics:')
display(africa_df['fire_pixels'].describe())

Building voting masks:   0%|          | 0/27729 [00:00<?, ?it/s]

Total African patches: 27729

Label distribution:


sample_type
voting_fire    22500
voting_zero     5229
Name: count, dtype: int64


Algorithm-mask coverage:


n_algorithm_masks
1     5138
2     4906
3    17685
Name: count, dtype: int64


Fire-pixel statistics:


count    27729.000000
mean         8.854376
std         24.357180
min          0.000000
25%          1.000000
50%          3.000000
75%          8.000000
max        630.000000
Name: fire_pixels, dtype: float64

In [6]:
# ================================================================
# CELL 6 — SCENE-LEVEL SPLIT
# ================================================================

df_all = africa_df.copy()

df_all['parent_scene'] = (
    df_all['stem']
    .str.replace(r'_p\d+$', '', regex=True)
)

rng = np.random.default_rng(SEED)
scenes = df_all['parent_scene'].drop_duplicates().to_numpy()
assert len(scenes) >= 3, (
    f'Need at least 3 independent scenes for train/val/test; found {len(scenes)}.'
)
rng.shuffle(scenes)

n_test = max(1, round(0.15 * len(scenes)))
n_val = max(1, round(0.15 * len(scenes)))

# Keep at least one independent scene in the training split even for small releases.
while n_test + n_val >= len(scenes):
    if n_val > 1:
        n_val -= 1
    elif n_test > 1:
        n_test -= 1
    else:
        raise RuntimeError('Unable to create non-empty train/val/test scene splits.')

test_scenes = set(scenes[:n_test])
val_scenes = set(scenes[n_test:n_test + n_val])
train_scenes = set(scenes[n_test + n_val:])

assert train_scenes and val_scenes and test_scenes
assert not (train_scenes & val_scenes)
assert not (train_scenes & test_scenes)
assert not (val_scenes & test_scenes)


def split_of(scene):
    if scene in train_scenes:
        return 'train'
    if scene in val_scenes:
        return 'val'
    return 'test'


df_all['split'] = df_all['parent_scene'].map(split_of)

train_df = df_all[df_all['split'] == 'train'].copy()
val_df = df_all[df_all['split'] == 'val'].copy()
test_df = df_all[df_all['split'] == 'test'].copy()

print(f'Train: {len(train_df):,} patches / {train_df.parent_scene.nunique():,} scenes')
print(f'Val:   {len(val_df):,} patches / {val_df.parent_scene.nunique():,} scenes')
print(f'Test:  {len(test_df):,} patches / {test_df.parent_scene.nunique():,} scenes')
print(f'Approximate scene split: {len(train_scenes)/len(scenes):.1%} / {len(val_scenes)/len(scenes):.1%} / {len(test_scenes)/len(scenes):.1%}')

assert len(train_df) and len(val_df) and len(test_df)

split_summary = (
    df_all.groupby(['split', 'sample_type'])
    .size()
    .unstack(fill_value=0)
)
display(split_summary)

split_summary.to_csv(OUTPUT_DIR / 'split_summary_v7.csv')
df_all.to_csv(OUTPUT_DIR / 'africa_inventory_split_v7_pre_cache.csv', index=False)


Train: 18,931 patches / 360 scenes
Val:   4,017 patches / 77 scenes
Test:  4,781 patches / 77 scenes
Approximate scene split: 70.0% / 15.0% / 15.0%


sample_type,voting_fire,voting_zero
split,,
test,3946,835
train,15377,3554
val,3177,840


In [7]:
# ================================================================
# CELL 7 — COMPACT MULTIBAND CACHE (read every TIFF ONCE)
# ================================================================
# Reading 10-band 16-bit TIFFs on every epoch was the bottleneck of V6 (and the
# cause of the worker crashes). We store only the 4 bands we need (c2, c6, c7, B10,
# raw uint16 DN) + the voting mask in two memory-mapped .npy files (~16 GB total).
# Training, evaluation, probes and the pipeline all read from this cache.

CACHE_BAND_NAMES = [n for n, _ in CACHE_BANDS]
CACHE_BAND_NUMBERS = [b for _, b in CACHE_BANDS]
PATCH_SIZE = 256

df_all = df_all.sort_values('stem').reset_index(drop=True)
df_all['cache_idx'] = np.arange(len(df_all))
N_ALL = len(df_all)

bands_path = CACHE_DIR / 'bands_u16.npy'
masks_path = CACHE_DIR / 'masks_u8.npy'
marker_path = CACHE_DIR / 'cache_ok.json'
geo_path = CACHE_DIR / 'geo_test.npy'
signature = hashlib.md5('|'.join(df_all['stem']).encode()).hexdigest()


def _pad_to(arr, size=PATCH_SIZE):
    """Pad [..., h, w] with zeros to size x size (border patches); returns (array, was_padded)."""
    h, w = arr.shape[-2:]
    if h == size and w == size:
        return arr, False
    out = np.zeros(arr.shape[:-2] + (size, size), dtype=arr.dtype)
    out[..., :min(h, size), :min(w, size)] = arr[..., :size, :size]
    return out, True


def read_patch_bands(path, band_numbers):
    with rasterio.open(path) as src:
        arr = src.read(list(band_numbers))
    return _pad_to(arr)


def read_patch_mask(path):
    with rasterio.open(path) as src:
        arr = (src.read(1) > 0).astype(np.uint8)
    return _pad_to(arr)


def build_cache(df, bands_file, masks_file, band_numbers,
                read_bands=read_patch_bands, read_mask=read_patch_mask, n_threads=CACHE_THREADS):
    n = len(df)
    X = np.lib.format.open_memmap(bands_file, mode='w+', dtype=np.uint16,
                                  shape=(n, len(band_numbers), PATCH_SIZE, PATCH_SIZE))
    Y = np.lib.format.open_memmap(masks_file, mode='w+', dtype=np.uint8,
                                  shape=(n, PATCH_SIZE, PATCH_SIZE))
    img_paths = df['image_path'].tolist()
    mask_paths = df['mask_path'].tolist()
    padded = []

    def work(i):
        arr, p1 = read_bands(img_paths[i], band_numbers)
        msk, p2 = read_mask(mask_paths[i])
        X[i] = arr.astype(np.uint16, copy=False)
        Y[i] = msk
        return i if (p1 or p2) else None

    if n_threads <= 1:
        results = [work(i) for i in tqdm(range(n), desc='Caching patches')]
    else:
        with ThreadPoolExecutor(max_workers=n_threads) as ex:
            results = list(tqdm(ex.map(work, range(n)), total=n, desc='Caching patches'))
    padded = [r for r in results if r is not None]
    X.flush()
    Y.flush()
    del X, Y
    return len(padded)


cache_ready = (
    marker_path.exists() and bands_path.exists() and masks_path.exists()
    and json.loads(marker_path.read_text()).get('signature') == signature
)

if cache_ready:
    print('✅ Reusing existing cache:', bands_path)
else:
    t0 = time.time()
    n_padded = build_cache(df_all, bands_path, masks_path, CACHE_BAND_NUMBERS)
    marker_path.write_text(json.dumps({
        'signature': signature, 'n': N_ALL, 'bands': CACHE_BAND_NAMES,
        'band_numbers': CACHE_BAND_NUMBERS, 'n_padded_border_patches': n_padded,
    }))
    print(f'✅ Cache built in {(time.time() - t0) / 60:.1f} min | border patches padded: {n_padded}')

BANDS_MM = np.load(bands_path, mmap_mode='r')    # [N, 4, 256, 256] uint16
MASKS_MM = np.load(masks_path, mmap_mode='r')    # [N, 256, 256]    uint8
assert BANDS_MM.shape[0] == N_ALL and MASKS_MM.shape[0] == N_ALL

# ---- verify the cache against direct TIFF reads (only possible while the patches exist) ----
if PATCH_DIR.exists() and any(PATCH_DIR.glob('*.tif')):
    rng_v = np.random.default_rng(SEED)
    for i in rng_v.choice(N_ALL, size=25, replace=False):
        arr, _ = read_patch_bands(df_all.loc[i, 'image_path'], CACHE_BAND_NUMBERS)
        msk, _ = read_patch_mask(df_all.loc[i, 'mask_path'])
        assert np.array_equal(arr, BANDS_MM[i]), f'cache mismatch (bands) at row {i}'
        assert np.array_equal(msk, MASKS_MM[i]), f'cache mismatch (mask) at row {i}'
        assert int(msk.sum()) == int(df_all.loc[i, 'fire_pixels']), 'fire_pixels mismatch'
    print('✅ 25 random patches verified against the original TIFFs')

# ---- georeferencing (test split only): lat/lon of 3 reference pixels -> linear model ----
# lat(x,y) = lat0 + dlat_dx*x + dlat_dy*y  (error is negligible over a 7.7 km patch).
# Flight geolocation will instead come from ephemeris + attitude; this is a ground-side stand-in.
def compute_geo(image_path):
    try:
        with rasterio.open(image_path) as src:
            if src.crs is None:
                return [np.nan] * 6
            cols = [0, PATCH_SIZE - 1, 0]
            rows = [0, 0, PATCH_SIZE - 1]
            xs, ys = rasterio.transform.xy(src.transform, rows, cols, offset='center')
            lon, lat = rio_transform(src.crs, 'EPSG:4326', list(xs), list(ys))
        d = PATCH_SIZE - 1
        return [lat[0], lon[0], (lat[1] - lat[0]) / d, (lon[1] - lon[0]) / d,
                (lat[2] - lat[0]) / d, (lon[2] - lon[0]) / d]
    except Exception:
        return [np.nan] * 6


def pixel_to_latlon(geo_row, x, y):
    """geo_row = [lat0, lon0, dlat_dx, dlon_dx, dlat_dy, dlon_dy]; returns (lat, lon) or (nan, nan)."""
    if geo_row is None or not np.all(np.isfinite(geo_row)):
        return float('nan'), float('nan')
    lat0, lon0, dlat_dx, dlon_dx, dlat_dy, dlon_dy = [float(v) for v in geo_row]
    return lat0 + dlat_dx * x + dlat_dy * y, lon0 + dlon_dx * x + dlon_dy * y


if geo_path.exists():
    GEO = np.load(geo_path)
else:
    GEO = np.full((N_ALL, 6), np.nan, dtype=np.float64)
    test_rows = df_all[df_all['split'] == 'test']
    if PATCH_DIR.exists():
        for i, p in tqdm(zip(test_rows['cache_idx'], test_rows['image_path']),
                         total=len(test_rows), desc='Geo (test split)'):
            GEO[i] = compute_geo(p)
        np.save(geo_path, GEO)
print('Geo rows available:', int(np.isfinite(GEO[:, 0]).sum()))

# ---- re-derive the split frames now that every row has a cache index ----
train_df = df_all[df_all['split'] == 'train'].copy()
val_df = df_all[df_all['split'] == 'val'].copy()
test_df = df_all[df_all['split'] == 'test'].copy()
df_all.to_csv(OUTPUT_DIR / 'africa_inventory_split_v7.csv', index=False)
print(f'Train {len(train_df):,} | Val {len(val_df):,} | Test {len(test_df):,}')

# ---- free the 36 GB of extracted patches (originals stay in /kaggle/input) ----
if FREE_PATCHES_AFTER_CACHE and PATCH_DIR.exists():
    shutil.rmtree(PATCH_DIR, ignore_errors=True)
    print('Freed extracted TIFF patches. Do NOT re-run CELL 4/5 now (they would re-extract them).')
!du -sh /kaggle/tmp/fireedge 2>/dev/null


Caching patches:   0%|          | 0/27729 [00:00<?, ?it/s]

✅ Cache built in 4.6 min | border patches padded: 0
✅ 25 random patches verified against the original TIFFs


Geo (test split):   0%|          | 0/4781 [00:00<?, ?it/s]

Geo rows available: 4781
Train 18,931 | Val 4,017 | Test 4,781
Freed extracted TIFF patches. Do NOT re-run CELL 4/5 now (they would re-extract them).
50G	/kaggle/tmp/fireedge


In [8]:
# ================================================================
# CELL 8 — PREPROCESSING CONTRACT + DATASET + LOADERS
# ================================================================
# ONE preprocessing function is used for training, evaluation, ONNX calibration and
# the onboard pipeline, so there is no train/deploy skew.
#   reflective bands (c2, c6, c7): Landsat-8 reflectance rho = 2e-5*DN - 0.1, clipped to [0, 1.6]
#   thermal band (b10): brightness temperature (K) -> (T - 300)/30, clipped to [-3, 6]
REFL_SCALE, REFL_OFFSET, REFL_MAX = 2.0e-5, -0.1, 1.6
B10_ML, B10_AL, B10_K1, B10_K2 = 3.342e-4, 0.1, 774.8853, 1321.0789
B10_CENTER_K, B10_SCALE_K = 300.0, 30.0


def dn_to_reflectance(dn):
    return np.clip(dn.astype(np.float32) * REFL_SCALE + REFL_OFFSET, 0.0, REFL_MAX)


def dn_to_bt_kelvin(dn):
    radiance = np.maximum(dn.astype(np.float32) * B10_ML + B10_AL, 1e-6)
    return (B10_K2 / np.log(B10_K1 / radiance + 1.0)).astype(np.float32)


def preprocess_dn(dn, names):
    """dn: uint16/float [C,H,W] ordered like names -> float32 [C,H,W]."""
    out = np.empty(dn.shape, dtype=np.float32)
    for k, name in enumerate(names):
        if name == 'b10':
            out[k] = np.clip((dn_to_bt_kelvin(dn[k]) - B10_CENTER_K) / B10_SCALE_K, -3.0, 6.0)
        else:
            out[k] = dn_to_reflectance(dn[k])
    return out


def channel_slots(names):
    return [CACHE_BAND_NAMES.index(n) for n in names]


class BandDataset(Dataset):
    """Reads raw DN from the memory-mapped cache and applies shared preprocessing."""

    def __init__(self, df, names, augment=False, seed=SEED):
        self.rows = df['cache_idx'].to_numpy()
        self.names = list(names)
        self.slots = channel_slots(names)
        self.augment = augment
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, k):
        i = int(self.rows[k])
        dn = np.asarray(BANDS_MM[i][self.slots])
        m = np.asarray(MASKS_MM[i])
        if self.augment:
            if self.rng.random() < 0.5:
                dn, m = dn[:, :, ::-1], m[:, ::-1]
            if self.rng.random() < 0.5:
                dn, m = dn[:, ::-1, :], m[::-1, :]
            r = int(self.rng.integers(0, 4))
            if r:
                dn, m = np.rot90(dn, r, axes=(1, 2)), np.rot90(m, r)
        x = preprocess_dn(np.ascontiguousarray(dn), self.names)
        y = (np.ascontiguousarray(m) > 0).astype(np.float32)[None]
        return torch.from_numpy(x), torch.from_numpy(y)


BUCKET_NAMES = ['0 px', '1-5 px', '6-25 px', '26-100 px', '>100 px']
BUCKET_EDGES = [1, 6, 26, 101]


def make_sampler_weights(frame, target_share=SIZE_TARGET_SHARE):
    """Explicit target-share sampler that keeps 1–5 px fires prominent."""
    fire_px = frame['fire_pixels'].to_numpy()
    bucket = np.digitize(fire_px, BUCKET_EDGES)
    counts = np.bincount(bucket, minlength=5).astype(np.int64)

    target = np.asarray(target_share, dtype=np.float64).copy()
    present = counts > 0
    target[~present] = 0.0
    target /= max(target.sum(), 1e-12)

    w = np.zeros(len(frame), dtype=np.float64)
    for b in range(5):
        if counts[b] > 0:
            w[bucket == b] = target[b] / counts[b]
    return w, bucket, counts, target


def make_loaders(names):
    ds = {
        'train': BandDataset(train_df, names, augment=True),
        'val': BandDataset(val_df, names),
        'test': BandDataset(test_df, names),
    }
    weights, _, _, _ = make_sampler_weights(train_df)
    sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double),
                                    num_samples=len(ds['train']), replacement=True)
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    return {
        'train': DataLoader(ds['train'], sampler=sampler, **kw),
        'val': DataLoader(ds['val'], shuffle=False, **kw),
        'test': DataLoader(ds['test'], shuffle=False, **kw),
    }


# ---------------- sanity checks ----------------
w_, bucket_, counts_, target_ = make_sampler_weights(train_df)
share_nat = counts_ / max(counts_.sum(), 1)
share_eff = np.array([w_[bucket_ == b].sum() for b in range(5)], dtype=np.float64)
share_eff /= max(share_eff.sum(), 1e-12)
print('========== TRAIN SAMPLER (EXPLICIT TARGET SHARE) ==========')
print(pd.DataFrame({'bucket': BUCKET_NAMES, 'patches': counts_,
                    'natural_share': np.round(share_nat, 4),
                    'target_share': np.round(target_, 4),
                    'sampled_share': np.round(share_eff, 4)}).to_string(index=False))

_probe = BandDataset(train_df.head(4), BAND_CONFIGS['pereira3c_b10'])
_x, _y = _probe[0]
assert _x.shape == (4, 256, 256) and _y.shape == (1, 256, 256)
assert torch.isfinite(_x).all() and set(torch.unique(_y).tolist()) <= {0.0, 1.0}
print('\nSample tensor:', tuple(_x.shape), 'per-channel min/max:',
      [(round(float(c.min()), 3), round(float(c.max()), 3)) for c in _x])
print('B10 check: DN 30397 ->', round(float(dn_to_bt_kelvin(np.array([30397]))[0]), 2), 'K')

========== TRAIN SAMPLER (EXPLICIT TARGET SHARE) ==========
   bucket  patches  natural_share  target_share  sampled_share
     0 px     3554         0.1877          0.15           0.15
   1-5 px     9307         0.4916          0.45           0.45
  6-25 px     4596         0.2428          0.25           0.25
26-100 px     1262         0.0667          0.10           0.10
  >100 px      212         0.0112          0.05           0.05

Sample tensor: (4, 256, 256) per-channel min/max: [(0.025, 0.494), (0.05, 0.555), (0.089, 0.639), (-0.514, 0.152)]
B10 check: DN 30397 -> 304.55 K


In [9]:
# ================================================================
# CELL 9 — MODEL, LOSS, METRICS, CALIBRATION, THRESHOLD TOOLS  (V7 final)
# ================================================================
from scipy.special import expit
from scipy.optimize import minimize_scalar

THRESHOLD_GRID = np.round(np.arange(0.05, 0.9901, 0.01), 2)
HIST_BINS = 1000

FAR_SELECTION_MARGIN = 0.03    # select thresholds on FAR <= MAX_FRAME_FAR - 0.03 (estimation-noise margin)
PLATEAU_F1_TOL = 0.005         # take the CENTRE of the F1 plateau (within 0.005 of the best), not the arg-max
TVERSKY_SMOOTH_PX = 1.0        # smoothing in PIXEL units (was 1e-6). Set 1e-6 to reproduce the old loss.

EVENT_AREA_CLASSES = [(1, 5, '1-5 px'), (6, 25, '6-25 px'), (26, 100, '26-100 px'), (101, 10**9, '>100 px')]
EVENT_CONF_TABLE = None        # filled by CELL 12B (event-level calibration); None -> mean calibrated probability


# ================================================================
# MODEL
# ================================================================

class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                cin,
                cout,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                cout,
                cout,
                3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TinyUNet(nn.Module):
    """
    3-level lightweight U-Net.

    Input:
        [B, C, 256, 256]

    Output:
        [B, 1, 256, 256]

    C = number of input bands.
    """

    def __init__(self, in_ch=3, base=16):
        super().__init__()

        # ---------------- encoder ----------------
        self.e1 = ConvBlock(in_ch, base)
        self.e2 = ConvBlock(base, base * 2)
        self.e3 = ConvBlock(base * 2, base * 4)

        self.pool = nn.MaxPool2d(2)

        # ---------------- bottleneck ----------------
        self.mid = ConvBlock(base * 4, base * 8)

        # ---------------- decoder ----------------
        self.u3 = nn.ConvTranspose2d(
            base * 8,
            base * 4,
            kernel_size=2,
            stride=2
        )

        self.d3 = ConvBlock(
            base * 8,
            base * 4
        )

        self.u2 = nn.ConvTranspose2d(
            base * 4,
            base * 2,
            kernel_size=2,
            stride=2
        )

        self.d2 = ConvBlock(
            base * 4,
            base * 2
        )

        self.u1 = nn.ConvTranspose2d(
            base * 2,
            base,
            kernel_size=2,
            stride=2
        )

        self.d1 = ConvBlock(
            base * 2,
            base
        )

        # ---------------- output ----------------
        self.out = nn.Conv2d(
            base,
            1,
            kernel_size=1
        )

    def forward(self, x):
        # Encoder
        e1 = self.e1(x)

        e2 = self.e2(
            self.pool(e1)
        )

        e3 = self.e3(
            self.pool(e2)
        )

        # Bottleneck
        m = self.mid(
            self.pool(e3)
        )

        # Decoder
        d3 = self.d3(
            torch.cat(
                [
                    self.u3(m),
                    e3
                ],
                dim=1
            )
        )

        d2 = self.d2(
            torch.cat(
                [
                    self.u2(d3),
                    e2
                ],
                dim=1
            )
        )

        d1 = self.d1(
            torch.cat(
                [
                    self.u1(d2),
                    e1
                ],
                dim=1
            )
        )

        return self.out(d1)


# ================================================================
# MODEL SIZE / COMPUTE
# ================================================================

def count_params(model):
    return int(
        sum(
            p.numel()
            for p in model.parameters()
        )
    )


def count_macs(model, in_ch, size=256):
    """
    Count multiply-accumulates for Conv2d and ConvTranspose2d
    for one input frame.
    """

    total = [0]

    def conv_hook(module, inputs, output):
        total[0] += (
            output.shape[2]
            * output.shape[3]
            * module.out_channels
            * (module.in_channels // module.groups)
            * module.kernel_size[0]
            * module.kernel_size[1]
        )

    def convt_hook(module, inputs, output):
        total[0] += (
            inputs[0].shape[2]
            * inputs[0].shape[3]
            * module.in_channels
            * (module.out_channels // module.groups)
            * module.kernel_size[0]
            * module.kernel_size[1]
        )

    hooks = []

    for module in model.modules():

        if isinstance(
            module,
            nn.ConvTranspose2d
        ):
            hooks.append(
                module.register_forward_hook(
                    convt_hook
                )
            )

        elif isinstance(
            module,
            nn.Conv2d
        ):
            hooks.append(
                module.register_forward_hook(
                    conv_hook
                )
            )

    was_training = model.training

    model.eval()

    with torch.no_grad():

        dummy = torch.zeros(
            1,
            in_ch,
            size,
            size,
            device=next(model.parameters()).device
        )

        model(dummy)

    for h in hooks:
        h.remove()

    model.train(was_training)

    return int(total[0])

# ================================================================
# LOSS  (FP32; non-finite checks are done by the training loop, not here)
# ================================================================
class FocalTverskyLoss(nn.Module):
    """Per-image focal Tversky.  Smoothing is in pixel units: with 1e-6 an EMPTY frame has loss ~1 and
    ~zero gradient for any realistic false-positive mass, and tv can hit exactly 1 -> pow(0, 0.75) -> NaN.
    With smooth=1 an empty frame gets a graded, finite false-positive penalty."""

    def __init__(self, alpha=0.3, beta=0.7, gamma=0.75, smooth=TVERSKY_SMOOTH_PX):
        super().__init__()
        self.alpha, self.beta, self.gamma, self.smooth = alpha, beta, gamma, smooth

    def forward(self, logits, targets):
        p = torch.sigmoid(logits.float())
        t = targets.float()
        dims = (1, 2, 3)
        tp = (p * t).sum(dims)
        fp = (p * (1.0 - t)).sum(dims)
        fn = ((1.0 - p) * t).sum(dims)
        tv = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        return (1.0 - tv).clamp(min=1e-6).pow(self.gamma).mean()


loss_tversky = FocalTverskyLoss()


def make_loss(pos_weight=POS_WEIGHT):
    bce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=device))

    def loss_fn(logits, targets):
        logits = logits.float()
        targets = targets.float()
        return 0.3 * bce(logits, targets) + 0.7 * loss_tversky(logits, targets)

    return loss_fn


# ================================================================
# PIXEL METRICS
# ================================================================

def metrics_from_counts(tp, fp, fn):

    return {
        'precision': (
            tp
            / (tp + fp + 1e-9)
        ),

        'recall': (
            tp
            / (tp + fn + 1e-9)
        ),

        'f1': (
            2 * tp
            / (
                2 * tp
                + fp
                + fn
                + 1e-9
            )
        ),

        'iou': (
            tp
            / (
                tp
                + fp
                + fn
                + 1e-9
            )
        ),

        'tp': tp,
        'fp': fp,
        'fn': fn,
    }


# ================================================================
# CALIBRATION HELPERS
# ================================================================
def sigmoid_temperature_np(logits, temperature=1.0):
    z = np.asarray(logits, dtype=np.float64) / max(float(temperature), 1e-3)
    return expit(np.clip(z, -60.0, 60.0)).astype(np.float32)


@torch.no_grad()
def fit_temperature(model, loader, neg_per_frame=1500, seed=SEED, t_bounds=(0.25, 20.0)):
    """Scalar temperature T minimising the NATURAL-distribution pixel BCE of sigmoid(z/T).
    Negatives are sub-sampled per frame and re-weighted so the fit is not biased by the sub-sampling.
    Returns (T, n_pixels_used)."""
    model.eval()
    rng = np.random.default_rng(seed)
    zs, ys, ws = [], [], []
    for images, masks in loader:
        z = model(images.to(device, non_blocking=True)).float().cpu().numpy()
        g = masks.numpy() > 0.5
        z = z.reshape(z.shape[0], -1)
        g = g.reshape(g.shape[0], -1)
        for zi, gi in zip(z, g):
            pos = np.flatnonzero(gi)
            neg = np.flatnonzero(~gi)
            if len(neg) > neg_per_frame:
                sel, w_neg = rng.choice(neg, neg_per_frame, replace=False), len(neg) / neg_per_frame
            else:
                sel, w_neg = neg, 1.0
            zs += [zi[pos], zi[sel]]
            ys += [np.ones(len(pos)), np.zeros(len(sel))]
            ws += [np.ones(len(pos)), np.full(len(sel), w_neg)]
    z = np.clip(np.concatenate(zs).astype(np.float64), -60.0, 60.0)
    y = np.concatenate(ys)
    w = np.concatenate(ws)

    def nll(log_t):
        s = z / np.exp(log_t)
        return float(np.sum(w * (np.logaddexp(0.0, s) - y * s)) / np.sum(w))

    res = minimize_scalar(nll, bounds=(np.log(t_bounds[0]), np.log(t_bounds[1])), method='bounded')
    return float(np.exp(res.x)), int(len(z))


# ================================================================
# EVALUATION (exact, FP32)
# ================================================================
@torch.no_grad()
def evaluate(model, loader, loss_fn, threshold, temperature=1.0):
    model.eval()
    losses, tp, fp, fn = [], 0, 0, 0
    fire_frames = no_fire_frames = correct = false_alarm = 0
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(images).float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError('Non-finite logits detected during evaluation.')
        losses.append(float(loss_fn(logits, masks).item()))
        pred = torch.sigmoid(logits / float(temperature)) >= threshold
        gt = masks > 0.5
        tp += int((pred & gt).sum().item())
        fp += int((pred & ~gt).sum().item())
        fn += int((~pred & gt).sum().item())
        gt_f = gt.flatten(1).any(1)
        pd_f = pred.flatten(1).any(1)
        fire_frames += int(gt_f.sum().item())
        no_fire_frames += int((~gt_f).sum().item())
        correct += int((gt_f & pd_f).sum().item())
        false_alarm += int((~gt_f & pd_f).sum().item())
    out = metrics_from_counts(tp, fp, fn)
    out.update(loss=float(np.mean(losses)), frame_recall=correct / max(fire_frames, 1),
               frame_false_alarm_rate=false_alarm / max(no_fire_frames, 1),
               fire_frames=fire_frames, no_fire_frames=no_fire_frames)
    return out


# ================================================================
# HISTOGRAM THRESHOLD SWEEP  (bin index and threshold now use the SAME mapping)
# ================================================================
def new_stats_state(bins=HIST_BINS):
    return {'pos': np.zeros(bins, np.int64), 'neg': np.zeros(bins, np.int64),
            'frame_max': [], 'frame_gt': [], 'losses': []}


def accumulate_stats_np(prob, gt, state, bins=HIST_BINS):
    prob = np.asarray(prob, dtype=np.float32)
    if not np.isfinite(prob).all():
        raise FloatingPointError('Non-finite probabilities reached accumulate_stats_np().')
    prob = np.clip(prob, 0.0, 1.0)
    idx = np.minimum((prob * bins).astype(np.int64), bins - 1)   # p >= k/bins  <=>  idx >= k
    g = np.asarray(gt, dtype=bool)
    state['pos'] += np.bincount(idx[g], minlength=bins)
    state['neg'] += np.bincount(idx[~g], minlength=bins)
    state['frame_max'].append(prob.reshape(prob.shape[0], -1).max(1))
    state['frame_gt'].append(g.reshape(g.shape[0], -1).any(1))


@torch.no_grad()
def collect_stats(model, loader, loss_fn, temperature=1.0):
    model.eval()
    state = new_stats_state()
    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(images).float()
        if not torch.isfinite(logits).all():
            raise FloatingPointError('Non-finite logits detected during validation.')
        state['losses'].append(float(loss_fn(logits, masks).item()))   # loss on RAW logits (training objective)
        prob = torch.sigmoid(logits / float(temperature)).cpu().numpy()
        accumulate_stats_np(prob, masks.cpu().numpy() > 0.5, state)
    return state


def sweep_from_stats(state, thresholds=THRESHOLD_GRID, bins=HIST_BINS):
    pos_cum = np.concatenate([np.cumsum(state['pos'][::-1])[::-1], [0]])   # pos_cum[k] = #pixels with bin >= k
    neg_cum = np.concatenate([np.cumsum(state['neg'][::-1])[::-1], [0]])
    n_pos = int(pos_cum[0])
    fmax = np.concatenate(state['frame_max'])
    fgt = np.concatenate(state['frame_gt'])
    n_fire, n_free = int(fgt.sum()), int((~fgt).sum())
    loss = float(np.mean(state['losses']))
    rows = []
    for t in thresholds:
        kb = min(max(int(round(float(t) * bins)), 0), bins)
        tp, fp = int(pos_cum[kb]), int(neg_cum[kb])
        m = metrics_from_counts(tp, fp, n_pos - tp)
        det = fmax >= float(t)
        rows.append({'threshold': float(t), 'loss': loss,
                     'precision': m['precision'], 'recall': m['recall'], 'f1': m['f1'], 'iou': m['iou'],
                     'tp': tp, 'fp': fp, 'fn': n_pos - tp,
                     'frame_recall': float((det & fgt).sum() / max(n_fire, 1)),
                     'frame_false_alarm_rate': float((det & ~fgt).sum() / max(n_free, 1)),
                     'fire_frames': n_fire, 'no_fire_frames': n_free})
    return pd.DataFrame(rows)


def select_operating_point(sweep, far_limit=None, min_frame_recall=None, plateau_tol=PLATEAU_F1_TOL):
    """Threshold for ONE model.
    1) feasible set  : frame_recall >= target AND FAR <= (limit - margin)
    2) else          : FAR <= (limit - margin), best F1
    3) else          : minimum normalised constraint violation
    Inside the chosen pool: centre of the F1 plateau (F1 within `plateau_tol` of the best)."""
    far_limit = (MAX_FRAME_FAR - FAR_SELECTION_MARGIN) if far_limit is None else far_limit
    min_fr = MIN_FRAME_RECALL if min_frame_recall is None else min_frame_recall
    sw = sweep.reset_index(drop=True)
    ok = sw[(sw['frame_recall'] >= min_fr) & (sw['frame_false_alarm_rate'] <= far_limit)]
    if len(ok):
        pool, rule, met = ok, 'constrained_plateau_f1', True
    else:
        far_ok = sw[sw['frame_false_alarm_rate'] <= far_limit]
        if len(far_ok):
            pool, rule = far_ok, 'infeasible_fallback_far_limited_plateau_f1'
        else:
            viol = (np.maximum(0.0, min_fr - sw['frame_recall']) / max(min_fr, 1e-9)
                    + np.maximum(0.0, sw['frame_false_alarm_rate'] - far_limit) / max(far_limit, 1e-9))
            pool, rule = sw.loc[[int(viol.idxmin())]], 'infeasible_fallback_min_violation'
        met = False
    plateau = pool[pool['f1'] >= pool['f1'].max() - plateau_tol]
    thr_arr = plateau['threshold'].to_numpy()
    best = plateau.iloc[int(np.argmin(np.abs(thr_arr - np.median(thr_arr))))].copy()
    best['selection_rule'] = rule
    best['constraints_met'] = bool(met)
    best['far_limit_used'] = float(far_limit)
    best['plateau_lo'] = float(thr_arr.min())
    best['plateau_hi'] = float(thr_arr.max())
    best['frame_recall_gap_to_target'] = float(max(0.0, MIN_FRAME_RECALL - best['frame_recall']))
    best['far_margin_to_limit'] = float(MAX_FRAME_FAR - best['frame_false_alarm_rate'])
    return best


# ================================================================
# EVENT CONFIDENCE HOOK (default = mean calibrated probability; CELL 12B can install a fitted table)
# ================================================================
def apply_event_confidence(mean_p, area_px):
    cls_name = next(n for lo, hi, n in EVENT_AREA_CLASSES if lo <= int(area_px) <= hi)
    cls = EVENT_CONF_TABLE['classes'].get(cls_name) or EVENT_CONF_TABLE['classes']['all']
    return float(np.clip(np.interp(mean_p, cls['x'], cls['y']), 0.0, 1.0))


def event_confidence(prob, component, threshold):
    mean_p = float(np.mean(prob[component])) if component.any() else float(threshold)
    if EVENT_CONF_TABLE is None:
        return mean_p
    return apply_event_confidence(mean_p, int(component.sum()))


# ================================================================
# CHECKPOINT LOADER
# ================================================================
def load_model_from_checkpoint(
    path
):

    ck = torch.load(
        path,
        map_location=device,
        weights_only=False
    )

    model = TinyUNet(
        in_ch=int(
            ck['in_ch']
        ),
        base=int(
            ck['base']
        )
    ).to(device)

    model.load_state_dict(
        ck['model_state']
    )

    model.eval()

    return model, ck


# ================================================================
# MODEL CHARACTERISTICS REPORT
# ================================================================
for _cfg in RUN_CONFIGS:
    _m = TinyUNet(in_ch=len(BAND_CONFIGS[_cfg]), base=MODEL_BASE).to(device)
    print(f"{_cfg:14s} bands={BAND_CONFIGS[_cfg]}  params={count_params(_m):,}  "
          f"MACs/frame={count_macs(_m, len(BAND_CONFIGS[_cfg])) / 1e9:.3f} G")
    del _m


pereira3c      bands=['c7', 'c6', 'c2']  params=482,737  MACs/frame=2.319 G
b10            bands=['b10']  params=482,449  MACs/frame=2.301 G
pereira3c_b10  bands=['c7', 'c6', 'c2', 'b10']  params=482,881  MACs/frame=2.329 G


In [10]:
# ================================================================
# CELL 10 — BAND INFORMATION PROBE (no training, ~1 min, reads the cache)
# ================================================================
# How well does ONE band (local anomaly = pixel - 31x31 local mean) separate voting-fire
# pixels from the rest?  AUC 0.5 = nothing, 1.0 = perfect.  Justifies the band choice.
from sklearn.metrics import roc_auc_score, precision_recall_curve

PROBE_N, NEG_PER_PATCH = 1500, 2000
rng_p = np.random.default_rng(SEED)
probe_rows = val_df.sample(min(PROBE_N, len(val_df)), random_state=SEED)['cache_idx'].to_numpy()

scores = {n: [] for n in CACHE_BAND_NAMES}
labels, weights = [], []
for i in tqdm(probe_rows, desc='Probing bands'):
    arr = np.asarray(BANDS_MM[i]).astype(np.float32)
    m = np.asarray(MASKS_MM[i]) > 0
    fire_idx = np.flatnonzero(m.ravel())
    neg_all = np.flatnonzero(~m.ravel())
    neg_idx = rng_p.choice(neg_all, size=min(NEG_PER_PATCH, len(neg_all)), replace=False)
    take = np.concatenate([fire_idx, neg_idx])
    labels.append(np.concatenate([np.ones(len(fire_idx)), np.zeros(len(neg_idx))]))
    weights.append(np.concatenate([np.ones(len(fire_idx)),
                                   np.full(len(neg_idx), len(neg_all) / max(len(neg_idx), 1))]))
    for k, name in enumerate(CACHE_BAND_NAMES):
        a = arr[k] - ndimage.uniform_filter(arr[k], size=31, mode='reflect')
        scores[name].append(a.ravel()[take])

y, w = np.concatenate(labels), np.concatenate(weights)
rows = []
for name in CACHE_BAND_NAMES:
    s = np.concatenate(scores[name])
    prec, rec, _ = precision_recall_curve(y, s, sample_weight=w)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    rows.append({'band': name, 'pixel_AUC': roc_auc_score(y, s, sample_weight=w),
                 'best_single_threshold_pixel_F1': float(f1.max())})
probe_table = pd.DataFrame(rows).sort_values('pixel_AUC', ascending=False)
display(probe_table.round(3))
probe_table.to_csv(OUTPUT_DIR / 'band_information_probe.csv', index=False)
print('fire pixels probed:', int(y.sum()))


Probing bands:   0%|          | 0/1500 [00:00<?, ?it/s]

,band,pixel_AUC,best_single_threshold_pixel_F1
2,c7,0.993,0.638
3,b10,0.865,0.010
1,c6,0.820,0.156
0,c2,0.565,0.000


fire pixels probed: 11703


In [11]:
# ================================================================
# CELL 11 — TRAIN EACH BAND CONFIGURATION  (V7 final)
# ================================================================
# * AMP (fp16 forward) + GradScaler, SINGLE scaler.step()/update() per batch
#   (the previous notebook called both twice per batch = two optimizer steps).
# * Loss in FP32 (outside autocast), non-finite loss -> skip the batch and count it.
# * clip_grad_norm_(error_if_nonfinite=False): overflow is GradScaler's job.
# * GradScaler skips are COUNTED per epoch instead of printing one line per overflow.
#   ~1 skipped step per 2000 is normal: the scaler doubles its scale every 2000 clean
#   steps, hits the fp16 limit, skips ONE step and halves again.
# * EVERY epoch checkpoint is kept (CELL 12 picks the mission epoch on validation).
# ================================================================

TRAINED = {}
ALL_HISTORY = []
MAX_CONSECUTIVE_BAD_LOSS = 20

EPOCH_CKPT_ROOT = OUTPUT_DIR / 'epoch_checkpoints'
EPOCH_CKPT_ROOT.mkdir(parents=True, exist_ok=True)


def train_config(cfg_name):
    names = BAND_CONFIGS[cfg_name]
    loaders = make_loaders(names)

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    model = TinyUNet(in_ch=len(names), base=MODEL_BASE).to(device)
    loss_fn = make_loss(POS_WEIGHT)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=amp)

    cfg_dir = EPOCH_CKPT_ROOT / cfg_name
    cfg_dir.mkdir(parents=True, exist_ok=True)
    best_path = OUTPUT_DIR / f'unet_{cfg_name}_b{MODEL_BASE}_best.pt'

    best_f1, best_epoch, stale = -1.0, -1, 0
    epoch_paths = {}
    steps_per_epoch = len(loaders['train'])

    print('\n' + '=' * 76)
    print(f'TRAINING {cfg_name}: bands={names} | params={count_params(model):,} | '
          f'batch={BATCH_SIZE} | steps/epoch={steps_per_epoch} | AMP={amp}')
    print('=' * 76)

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        model.train()
        losses, grad_norms = [], []
        n_steps = n_overflow_skips = n_bad_loss = consecutive_bad = 0

        for images, masks in loaders['train']:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=amp, dtype=torch.float16):
                logits = model(images)
            loss = loss_fn(logits.float(), masks)          # FP32, outside autocast

            if not torch.isfinite(loss):
                n_bad_loss += 1
                consecutive_bad += 1
                if consecutive_bad >= MAX_CONSECUTIVE_BAD_LOSS:
                    raise FloatingPointError(
                        f'{consecutive_bad} consecutive non-finite losses at epoch {epoch}: training has diverged.')
                continue
            consecutive_bad = 0

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=False)
            gn = float(grad_norm)
            grad_ok = math.isfinite(gn)

            if amp:
                scale_before = scaler.get_scale()
                scaler.step(optimizer)          # skipped automatically when grads contain inf/nan
                scaler.update()
                if scaler.get_scale() < scale_before:
                    n_overflow_skips += 1
            else:
                if grad_ok:
                    optimizer.step()
                else:
                    n_overflow_skips += 1

            n_steps += 1
            losses.append(float(loss.item()))
            if grad_ok:
                grad_norms.append(gn)

        # ---------------- validation (raw score, T=1; calibration is done in CELL 12) ----------------
        sweep = sweep_from_stats(collect_stats(model, loaders['val'], loss_fn))
        best_row = sweep.loc[sweep['f1'].idxmax()]
        scheduler.step(float(best_row['loss']))

        row = {
            'config': cfg_name, 'epoch': epoch,
            'train_loss': float(np.mean(losses)) if losses else float('nan'),
            'val_loss': float(best_row['loss']), 'val_f1': float(best_row['f1']),
            'val_iou': float(best_row['iou']), 'val_precision': float(best_row['precision']),
            'val_recall': float(best_row['recall']), 'val_threshold_maxf1': float(best_row['threshold']),
            'lr': float(optimizer.param_groups[0]['lr']), 'epoch_seconds': time.time() - t0,
            'steps': n_steps, 'amp_overflow_skips': n_overflow_skips, 'bad_loss_batches': n_bad_loss,
            'grad_norm_mean': float(np.mean(grad_norms)) if grad_norms else float('nan'),
            'grad_norm_max': float(np.max(grad_norms)) if grad_norms else float('nan'),
            'amp_scale': float(scaler.get_scale()) if amp else 1.0,
        }

        ckpt_path = cfg_dir / f'unet_{cfg_name}_b{MODEL_BASE}_epoch{epoch:02d}.pt'
        ck = {
            'model_state': model.state_dict(), 'in_ch': len(names), 'base': MODEL_BASE,
            'channels': names, 'config': cfg_name, 'pos_weight': POS_WEIGHT, 'epoch': epoch,
            'val_f1_maxthr': row['val_f1'], 'val_threshold_maxf1': row['val_threshold_maxf1'],
            'preprocessing': {'refl_scale': REFL_SCALE, 'refl_offset': REFL_OFFSET, 'refl_max': REFL_MAX,
                              'b10_center_k': B10_CENTER_K, 'b10_scale_k': B10_SCALE_K},
            'training': {'optimizer': 'AdamW', 'lr0': 1e-3, 'weight_decay': 1e-4,
                         'loss': '0.3 BCE(pos_weight) + 0.7 FocalTversky(smooth_px=%s)' % TVERSKY_SMOOTH_PX,
                         'amp': amp, 'grad_clip': 1.0, 'seed': SEED},
        }
        torch.save(ck, ckpt_path)
        epoch_paths[epoch] = ckpt_path
        row['checkpoint'] = str(ckpt_path)
        ALL_HISTORY.append(row)

        print(f"ep{epoch:02d} train={row['train_loss']:.4f} val={row['val_loss']:.4f} "
              f"F1={row['val_f1']:.4f} IoU={row['val_iou']:.4f} P={row['val_precision']:.3f} "
              f"R={row['val_recall']:.3f} thr={row['val_threshold_maxf1']:.2f} "
              f"gradMean={row['grad_norm_mean']:.3f} gradMax={row['grad_norm_max']:.2f} "
              f"skips={n_overflow_skips}/{n_steps} badLoss={n_bad_loss} scale={row['amp_scale']:.0f} "
              f"({row['epoch_seconds']:.0f}s)")

        if row['val_f1'] > best_f1:
            best_f1, best_epoch, stale = row['val_f1'], epoch, 0
            torch.save(ck, best_path)
            print(f'  -> new best validation F1 (epoch {epoch})')
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f'Early stopping after {PATIENCE} epochs without validation-F1 improvement.')
                break

    TRAINED[cfg_name] = {'ckpt_dir': cfg_dir, 'names': names, 'ckpt': best_path,
                         'best_val_f1': best_f1, 'best_epoch': best_epoch, 'epoch_paths': epoch_paths}
    del model, optimizer, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


for cfg in RUN_CONFIGS:
    train_config(cfg)

history_df = pd.DataFrame(ALL_HISTORY)
history_df.to_csv(OUTPUT_DIR / 'training_history_v7.csv', index=False)
display(pd.DataFrame([{'config': c, 'bands': i['names'], 'best_val_f1': i['best_val_f1'],
                       'best_f1_epoch': i['best_epoch'], 'epochs_kept': len(i['epoch_paths'])}
                      for c, i in TRAINED.items()]))
print('\nNOTE: TRAINED[cfg]["ckpt"] is only the max-F1 epoch. CELL 12 replaces it with the mission-selected epoch.')



TRAINING pereira3c: bands=['c7', 'c6', 'c2'] | params=482,737 | batch=16 | steps/epoch=1184 | AMP=True
ep01 train=0.4091 val=0.0796 F1=0.9305 IoU=0.8700 P=0.931 R=0.930 thr=0.56 gradMean=0.630 gradMax=3.57 skips=0/1184 badLoss=0 scale=65536 (143s)
  -> new best validation F1 (epoch 1)
ep02 train=0.0816 val=0.0842 F1=0.9423 IoU=0.8908 P=0.960 R=0.925 thr=0.05 gradMean=1.146 gradMax=4.60 skips=1/1184 badLoss=0 scale=32768 (120s)
  -> new best validation F1 (epoch 2)
ep03 train=0.0715 val=0.0974 F1=0.9282 IoU=0.8661 P=0.970 R=0.890 thr=0.05 gradMean=1.170 gradMax=5.82 skips=0/1184 badLoss=0 scale=32768 (116s)
ep04 train=0.0664 val=0.0524 F1=0.9446 IoU=0.8951 P=0.934 R=0.955 thr=0.99 gradMean=1.223 gradMax=6.67 skips=1/1184 badLoss=0 scale=32768 (115s)
  -> new best validation F1 (epoch 4)
ep05 train=0.0662 val=0.1191 F1=0.9017 IoU=0.8210 P=0.986 R=0.831 thr=0.05 gradMean=1.203 gradMax=5.79 skips=1/1184 badLoss=0 scale=16384 (115s)
ep06 train=0.0635 val=0.0585 F1=0.9441 IoU=0.8941 P=0.940

,config,bands,best_val_f1,best_f1_epoch,epochs_kept
0,pereira3c,"[c7, c6, c2]",0.957502,19,20
1,b10,[b10],0.411990,17,20
2,pereira3c_b10,"[c7, c6, c2, b10]",0.955619,17,20



NOTE: TRAINED[cfg]["ckpt"] is only the max-F1 epoch. CELL 12 replaces it with the mission-selected epoch.


In [12]:
# ================================================================
# CELL 12 — MISSION-DRIVEN EPOCH + THRESHOLD SELECTION, CALIBRATION, HELD-OUT TEST  (V7 final)
# ================================================================
# Data hygiene (validation is split BY SCENE into two disjoint halves):
#   VAL-CAL : fits the scalar temperature T
#   VAL-SEL : chooses epoch + threshold          -> the TEST set is never used for selection
# Selection logic:
#   1. threshold per epoch  : constraints (frame recall >= target, FAR <= limit - 3 pts margin),
#                             centre of the F1 plateau (not the arg-max)
#   2. epoch ranking        : constraints met > event recall (location-matched, ALL sizes, rounded to 0.001)
#                             > fewer false events per fire-free patch > 1-5 px event recall > pixel F1
#   Only the TOP_K_EPOCHS epochs by validation F1 are evaluated (event matching is slow).
# ================================================================

EPOCH_CKPT_ROOT = globals().get('EPOCH_CKPT_ROOT', OUTPUT_DIR / 'epoch_checkpoints')
EVENT_TOL_PX = 3
EVENT_BUCKETS = EVENT_AREA_CLASSES
TOP_K_EPOCHS = 5
VAL_CAL_FRACTION = 0.4
MIN_VAL_SCENES_FOR_SPLIT = 4
_S8 = ndimage.generate_binary_structure(2, 2)


def match_events(pred, gt, tol=EVENT_TOL_PX):
    """pred, gt: bool [H,W] -> (gt_event_sizes, gt_event_found, n_pred_events, n_false_pred_events)"""
    g_lab, n_g = ndimage.label(gt, structure=_S8)
    p_lab, n_p = ndimage.label(pred, structure=_S8)
    sizes = np.bincount(g_lab.ravel())[1:] if n_g else np.zeros(0, int)
    if n_g:
        near = ndimage.binary_dilation(pred, structure=_S8, iterations=tol) if n_p else pred
        found = np.atleast_1d(ndimage.maximum(near.astype(np.uint8), g_lab, np.arange(1, n_g + 1))) > 0
    else:
        found = np.zeros(0, bool)
    n_false = 0
    if n_p:
        gnear = ndimage.binary_dilation(gt, structure=_S8, iterations=tol) if n_g else gt
        touched = np.atleast_1d(ndimage.maximum(gnear.astype(np.uint8), p_lab, np.arange(1, n_p + 1))) > 0
        n_false = int((~touched).sum())
    return sizes, found, n_p, n_false


@torch.no_grad()
def size_and_event_report(model, loader, threshold, temperature=1.0):
    model.eval()
    acc = [dict(frames=0, detected=0, tp=0, fp=0, fn=0) for _ in range(5)]
    ev = {name: [0, 0] for _, _, name in EVENT_BUCKETS}
    n_pred = n_false = n_frames = n_nofire = n_false_nofire = 0

    for images, masks in loader:
        z = model(images.to(device, non_blocking=True)).float().cpu().numpy()
        probs = sigmoid_temperature_np(z, temperature)[:, 0]
        gts = masks.numpy()[:, 0] > 0.5
        for p, g in zip(probs, gts):
            pred = p >= threshold
            b = int(np.digitize(int(g.sum()), BUCKET_EDGES))
            acc[b]['frames'] += 1
            acc[b]['detected'] += int(pred.any())
            acc[b]['tp'] += int((pred & g).sum())
            acc[b]['fp'] += int((pred & ~g).sum())
            acc[b]['fn'] += int((~pred & g).sum())
            sizes, found, npd, nfa = match_events(pred, g)
            for sz, f in zip(sizes, found):
                for lo, hi, name in EVENT_BUCKETS:
                    if lo <= sz <= hi:
                        ev[name][0] += 1
                        ev[name][1] += int(f)
            n_pred += npd
            n_false += nfa
            n_frames += 1
            if not g.any():
                n_nofire += 1
                n_false_nofire += nfa

    rows = []
    for k, name in enumerate(BUCKET_NAMES):
        a = acc[k]
        rows.append({
            'true_fire_pixels_per_patch': name, 'patches': a['frames'],
            'frame_recall': a['detected'] / max(a['frames'], 1) if k else np.nan,
            'false_alarm_rate': a['detected'] / max(a['frames'], 1) if k == 0 else np.nan,
            'pixel_recall': a['tp'] / max(a['tp'] + a['fn'], 1) if k else np.nan,
            'pixel_precision': a['tp'] / max(a['tp'] + a['fp'], 1) if k else np.nan,
            'pixel_f1': 2 * a['tp'] / max(2 * a['tp'] + a['fp'] + a['fn'], 1) if k else np.nan})
    ev_table = pd.DataFrame([{'true_event_size': name, 'events': n_e, 'event_recall': n_f / max(n_e, 1)}
                             for name, (n_e, n_f) in ev.items()])
    tot_e = sum(v[0] for v in ev.values())
    tot_f = sum(v[1] for v in ev.values())
    summary = {'event_precision': 1 - n_false / max(n_pred, 1),
               'event_recall_all': tot_f / max(tot_e, 1), 'n_gt_events': tot_e,
               'false_events_per_patch': n_false / max(n_frames, 1),
               'false_events_per_fire_free_patch': n_false_nofire / max(n_nofire, 1),
               'n_fire_free_patches': n_nofire}
    return pd.DataFrame(rows), ev_table, summary


# ---------------------------------------------------------------
# scene-level validation split
# ---------------------------------------------------------------
def make_frame_loader(frame, names):
    return DataLoader(BandDataset(frame, names), batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


def split_val_by_scene(frame, cal_fraction=VAL_CAL_FRACTION, seed=SEED):
    scenes = np.array(sorted(frame['parent_scene'].unique()))
    if len(scenes) < MIN_VAL_SCENES_FOR_SPLIT:
        print(f'WARNING: only {len(scenes)} validation scenes -> NOT splitting; the same validation set is used for '
              'T-fitting and selection (selection is then optimistic; keep the FAR margin).')
        return frame, frame, False
    rng = np.random.default_rng(seed + 11)
    rng.shuffle(scenes)
    n_cal = min(max(1, int(round(cal_fraction * len(scenes)))), len(scenes) - 1)
    cal_scenes = set(scenes[:n_cal])
    is_cal = frame['parent_scene'].isin(cal_scenes)
    return frame[~is_cal].copy(), frame[is_cal].copy(), True


VAL_SEL_DF, VAL_CAL_DF, VAL_SPLIT_OK = split_val_by_scene(val_df)
for _nm, _fr in (('VAL-SEL (epoch+threshold)', VAL_SEL_DF), ('VAL-CAL (temperature)', VAL_CAL_DF)):
    print(f"{_nm:28s}: {_fr['parent_scene'].nunique():3d} scenes | {len(_fr):6,} patches | "
          f"fire-free {(_fr['fire_pixels'] == 0).sum():6,}")


# ---------------------------------------------------------------
# per-config selection
# ---------------------------------------------------------------
def candidate_checkpoints(cfg):
    paths = sorted((EPOCH_CKPT_ROOT / cfg).glob(f'unet_{cfg}_b{MODEL_BASE}_epoch*.pt'))
    if not paths:
        raise FileNotFoundError(f'No per-epoch checkpoints for {cfg}: run CELL 11 first.')
    meta = []
    for p in paths:
        ck = torch.load(p, map_location='cpu', weights_only=False)
        meta.append((float(ck['val_f1_maxthr']), int(ck['epoch']), p))
    meta.sort(key=lambda m: m[0], reverse=True)
    return meta[:TOP_K_EPOCHS]


RESULTS, OPERATING, EPOCH_SELECTIONS = [], {}, {}

for cfg in RUN_CONFIGS:
    names = BAND_CONFIGS[cfg]
    loaders = make_loaders(names)
    loader_sel = make_frame_loader(VAL_SEL_DF, names)
    loader_cal = make_frame_loader(VAL_CAL_DF, names)
    loss_fn = make_loss(POS_WEIGHT)

    print('\n' + '=' * 76)
    print(f'MISSION SELECTION: {cfg}  (top {TOP_K_EPOCHS} epochs by validation F1)')
    print('=' * 76)

    cand_rows = []
    for f1_ref, epoch, ckpt_path in candidate_checkpoints(cfg):
        model, _ = load_model_from_checkpoint(ckpt_path)
        T, n_pix = fit_temperature(model, loader_cal)
        sweep = sweep_from_stats(collect_stats(model, loader_sel, loss_fn, T))
        op = select_operating_point(sweep)
        thr = float(op['threshold'])
        _, ev_t, ev_s = size_and_event_report(model, loader_sel, thr, T)
        small = ev_t[ev_t['true_event_size'] == '1-5 px']
        viol = (op['frame_recall_gap_to_target'] / max(MIN_FRAME_RECALL, 1e-9)
                + max(0.0, op['frame_false_alarm_rate'] - op['far_limit_used']) / max(op['far_limit_used'], 1e-9))
        r = {'config': cfg, 'epoch': epoch, 'checkpoint': str(ckpt_path), 'val_f1_maxthr_rawscore': f1_ref,
             'temperature': T, 'calibration_pixels': n_pix, 'threshold': thr, 'rule': op['selection_rule'],
             'constraints_met': bool(op['constraints_met']), 'violation': float(viol),
             'plateau_lo': float(op['plateau_lo']), 'plateau_hi': float(op['plateau_hi']),
             'frame_recall': float(op['frame_recall']), 'far': float(op['frame_false_alarm_rate']),
             'pixel_f1': float(op['f1']), 'pixel_recall': float(op['recall']),
             'event_recall_all': float(ev_s['event_recall_all']),
             'event_recall_1_5px': float(small.iloc[0]['event_recall']) if len(small) else np.nan,
             'false_events_per_fire_free_patch': float(ev_s['false_events_per_fire_free_patch']),
             'frame_recall_gap_to_target': float(op['frame_recall_gap_to_target']),
             'far_margin_to_limit': float(op['far_margin_to_limit'])}
        cand_rows.append(r)
        print(f"epoch {epoch:02d} | T={T:5.2f} thr={thr:.2f} [{r['plateau_lo']:.2f}-{r['plateau_hi']:.2f}] | "
              f"frameR={r['frame_recall']:.3f} FAR={r['far']:.3f} | eventR={r['event_recall_all']:.3f} "
              f"(1-5px {r['event_recall_1_5px']:.3f}) falseEv/free={r['false_events_per_fire_free_patch']:.3f} | "
              f"F1={r['pixel_f1']:.3f} | {'OK ' if r['constraints_met'] else 'INFEASIBLE '}{r['rule']}")
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    cand = pd.DataFrame(cand_rows)
    cand['rank_event_recall'] = cand['event_recall_all'].round(3)
    cand = cand.sort_values(
        ['constraints_met', 'violation', 'rank_event_recall', 'false_events_per_fire_free_patch',
         'event_recall_1_5px', 'pixel_f1'],
        ascending=[False, True, False, True, False, False], na_position='last').reset_index(drop=True)
    sel = cand.iloc[0]
    EPOCH_SELECTIONS[cfg] = cand
    cand.to_csv(OUTPUT_DIR / f'epoch_mission_selection_{cfg}.csv', index=False)

    sel_epoch, sel_ckpt = int(sel['epoch']), Path(sel['checkpoint'])
    T, thr = float(sel['temperature']), float(sel['threshold'])
    TRAINED.setdefault(cfg, {'names': names}).update({'ckpt': sel_ckpt, 'selected_epoch': sel_epoch})
    print(f"\n>>> SELECTED {cfg}: epoch {sel_epoch} | T={T:.3f} | threshold={thr:.2f} | "
          f"{'constraints met' if sel['constraints_met'] else 'INFEASIBLE (see rule)'}")

    # ---------------- held-out TEST (touched once, after selection) ----------------
    model, _ = load_model_from_checkpoint(sel_ckpt)
    sweep_sel = sweep_from_stats(collect_stats(model, loader_sel, loss_fn, T))
    sweep_sel.to_csv(OUTPUT_DIR / f'val_threshold_sweep_{cfg}.csv', index=False)
    (OUTPUT_DIR / f'probability_calibration_{cfg}.json').write_text(json.dumps({
        'temperature': T, 'calibration_pixels': int(sel['calibration_pixels']), 'selected_epoch': sel_epoch,
        'method': 'scalar logit temperature scaling on VAL-CAL pixels (scene-disjoint from VAL-SEL and TEST)',
        'val_split_scene_disjoint': bool(VAL_SPLIT_OK)}, indent=2))

    test_m = evaluate(model, loaders['test'], loss_fn, thr, T)
    test_025 = evaluate(model, loaders['test'], loss_fn, 0.25, T)
    by_size, ev_table, ev_sum = size_and_event_report(model, loaders['test'], thr, T)
    by_size.to_csv(OUTPUT_DIR / f'test_by_fire_size_{cfg}.csv', index=False)
    ev_table.to_csv(OUTPUT_DIR / f'test_event_recall_{cfg}.csv', index=False)

    OPERATING[cfg] = {
        'threshold': thr, 'temperature': T, 'calibration_pixels': int(sel['calibration_pixels']),
        'rule': sel['rule'], 'constraints_met': bool(sel['constraints_met']), 'selected_epoch': sel_epoch,
        'plateau': (float(sel['plateau_lo']), float(sel['plateau_hi'])),
        'frame_recall_gap_to_target': float(sel['frame_recall_gap_to_target']),
        'far_margin_to_limit': float(sel['far_margin_to_limit']),
        'by_size': by_size, 'events': ev_table, 'event_summary': ev_sum, 'test': test_m}

    row = {'config': cfg, 'selected_epoch': sel_epoch, 'bands': '+'.join(names), 'params': count_params(model),
           'MACs_G': count_macs(model, len(names)) / 1e9, 'temperature': T, 'threshold': thr,
           'rule': sel['rule'], 'constraints_met': bool(sel['constraints_met']),
           'F1': test_m['f1'], 'IoU': test_m['iou'], 'precision': test_m['precision'], 'recall': test_m['recall'],
           'frame_recall': test_m['frame_recall'], 'false_alarm_rate': test_m['frame_false_alarm_rate'],
           'F1@0.25': test_025['f1'], 'event_recall_all': ev_sum['event_recall_all'],
           'event_precision': ev_sum['event_precision'], 'false_events_per_patch': ev_sum['false_events_per_patch'],
           'false_events_per_fire_free_patch': ev_sum['false_events_per_fire_free_patch']}
    for _, r_ in ev_table.iterrows():
        row[f"event_recall {r_['true_event_size']}"] = r_['event_recall']
    RESULTS.append(row)

    print(f"[{cfg}] TEST | F1 {test_m['f1']:.3f} | frame recall {test_m['frame_recall']:.3f} | "
          f"FAR {test_m['frame_false_alarm_rate']:.3f} | event recall {ev_sum['event_recall_all']:.3f} | "
          f"false events / fire-free patch {ev_sum['false_events_per_fire_free_patch']:.3f}")
    display(by_size.round(3))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(RESULTS)
display(results_df.round(3))
results_df.to_csv(OUTPUT_DIR / 'test_results_by_config_v7.csv', index=False)

assert PRIMARY_CONFIG in TRAINED, 'PRIMARY_CONFIG must be one of the trained configurations.'
PRIMARY_NAMES = TRAINED[PRIMARY_CONFIG]['names']
PRIMARY_THRESHOLD = OPERATING[PRIMARY_CONFIG]['threshold']
PRIMARY_TEMPERATURE = OPERATING[PRIMARY_CONFIG]['temperature']
primary_model, _ = load_model_from_checkpoint(TRAINED[PRIMARY_CONFIG]['ckpt'])
primary_loss = make_loss(POS_WEIGHT)
print(f"\nPRIMARY: {PRIMARY_CONFIG} {PRIMARY_NAMES} | epoch {OPERATING[PRIMARY_CONFIG]['selected_epoch']} | "
      f"T={PRIMARY_TEMPERATURE:.3f} | threshold={PRIMARY_THRESHOLD:.2f} | "
      f"constraints_met={OPERATING[PRIMARY_CONFIG]['constraints_met']}")


VAL-SEL (epoch+threshold)   :  46 scenes |  2,169 patches | fire-free    452
VAL-CAL (temperature)       :  31 scenes |  1,848 patches | fire-free    388

MISSION SELECTION: pereira3c  (top 5 epochs by validation F1)
epoch 19 | T= 2.04 thr=0.94 [0.91-0.98] | frameR=0.956 FAR=0.066 | eventR=0.951 (1-5px 0.938) falseEv/free=0.069 | F1=0.955 | OK constrained_plateau_f1
epoch 13 | T= 1.95 thr=0.58 [0.58-0.59] | frameR=0.950 FAR=0.069 | eventR=0.946 (1-5px 0.932) falseEv/free=0.071 | F1=0.952 | OK constrained_plateau_f1
epoch 18 | T= 2.17 thr=0.65 [0.54-0.77] | frameR=0.956 FAR=0.069 | eventR=0.948 (1-5px 0.934) falseEv/free=0.071 | F1=0.952 | OK constrained_plateau_f1
epoch 14 | T= 2.15 thr=0.87 [0.79-0.95] | frameR=0.957 FAR=0.069 | eventR=0.953 (1-5px 0.941) falseEv/free=0.071 | F1=0.953 | OK constrained_plateau_f1
epoch 11 | T= 2.12 thr=0.96 [0.95-0.98] | frameR=0.957 FAR=0.066 | eventR=0.956 (1-5px 0.944) falseEv/free=0.069 | F1=0.951 | OK constrained_plateau_f1

>>> SELECTED pereira3c

,true_fire_pixels_per_patch,patches,frame_recall,false_alarm_rate,pixel_recall,pixel_precision,pixel_f1
0,0 px,835,NaN,0.063,NaN,NaN,NaN
1,1-5 px,2515,0.945,NaN,0.917,0.950,0.933
2,6-25 px,1098,1.000,NaN,0.948,0.956,0.952
3,26-100 px,303,1.000,NaN,0.961,0.949,0.955
4,>100 px,30,1.000,NaN,0.953,0.949,0.951



MISSION SELECTION: b10  (top 5 epochs by validation F1)
epoch 17 | T= 2.04 thr=0.20 [0.05-0.35] | frameR=0.171 FAR=0.020 | eventR=0.135 (1-5px 0.050) falseEv/free=0.029 | F1=0.405 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 19 | T= 2.02 thr=0.25 [0.06-0.44] | frameR=0.149 FAR=0.011 | eventR=0.125 (1-5px 0.048) falseEv/free=0.020 | F1=0.396 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 15 | T= 1.99 thr=0.17 [0.05-0.29] | frameR=0.178 FAR=0.018 | eventR=0.135 (1-5px 0.050) falseEv/free=0.031 | F1=0.397 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 12 | T= 1.97 thr=0.29 [0.05-0.54] | frameR=0.202 FAR=0.024 | eventR=0.142 (1-5px 0.055) falseEv/free=0.040 | F1=0.392 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 14 | T= 1.97 thr=0.32 [0.05-0.60] | frameR=0.149 FAR=0.015 | eventR=0.117 (1-5px 0.042) falseEv/free=0.022 | F1=0.388 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1

>>> SELECTED b10: epoch 12 | T=1.969 | thr

,true_fire_pixels_per_patch,patches,frame_recall,false_alarm_rate,pixel_recall,pixel_precision,pixel_f1
0,0 px,835,NaN,0.026,NaN,NaN,NaN
1,1-5 px,2515,0.044,NaN,0.017,0.108,0.029
2,6-25 px,1098,0.307,NaN,0.211,0.517,0.300
3,26-100 px,303,0.772,NaN,0.447,0.576,0.503
4,>100 px,30,1.000,NaN,0.618,0.638,0.628



MISSION SELECTION: pereira3c_b10  (top 5 epochs by validation F1)
epoch 17 | T= 2.13 thr=0.98 [0.98-0.99] | frameR=0.946 FAR=0.066 | eventR=0.941 (1-5px 0.925) falseEv/free=0.066 | F1=0.948 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 16 | T= 2.18 thr=0.60 [0.51-0.69] | frameR=0.955 FAR=0.069 | eventR=0.945 (1-5px 0.930) falseEv/free=0.071 | F1=0.951 | OK constrained_plateau_f1
epoch 18 | T= 2.05 thr=0.62 [0.36-0.88] | frameR=0.940 FAR=0.064 | eventR=0.939 (1-5px 0.923) falseEv/free=0.064 | F1=0.952 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1
epoch 14 | T= 2.03 thr=0.99 [0.99-0.99] | frameR=0.952 FAR=0.073 | eventR=0.949 (1-5px 0.936) falseEv/free=0.073 | F1=0.947 | INFEASIBLE infeasible_fallback_min_violation
epoch 09 | T= 1.93 thr=0.97 [0.97-0.98] | frameR=0.930 FAR=0.069 | eventR=0.920 (1-5px 0.899) falseEv/free=0.115 | F1=0.939 | INFEASIBLE infeasible_fallback_far_limited_plateau_f1

>>> SELECTED pereira3c_b10: epoch 16 | T=2.177 | threshold=0.60 | con

,true_fire_pixels_per_patch,patches,frame_recall,false_alarm_rate,pixel_recall,pixel_precision,pixel_f1
0,0 px,835,NaN,0.067,NaN,NaN,NaN
1,1-5 px,2515,0.944,NaN,0.891,0.959,0.924
2,6-25 px,1098,1.000,NaN,0.921,0.974,0.947
3,26-100 px,303,1.000,NaN,0.940,0.968,0.954
4,>100 px,30,1.000,NaN,0.943,0.964,0.953


,config,selected_epoch,bands,params,MACs_G,temperature,threshold,rule,constraints_met,F1,...,false_alarm_rate,F1@0.25,event_recall_all,event_precision,false_events_per_patch,false_events_per_fire_free_patch,event_recall 1-5 px,event_recall 6-25 px,event_recall 26-100 px,event_recall >100 px
0,pereira3c,11,c7+c6+c2,482737,2.319,2.123,0.96,constrained_plateau_f1,True,0.949,...,0.063,0.949,0.955,0.984,0.029,0.067,0.946,1.000,1.000,1.0
1,b10,12,b10,482449,2.301,1.969,0.29,infeasible_fallback_far_limited_plateau_f1,False,0.412,...,0.026,0.412,0.133,0.830,0.036,0.030,0.053,0.455,0.823,1.0
2,pereira3c_b10,16,c7+c6+c2+b10,482881,2.329,2.177,0.60,constrained_plateau_f1,True,0.946,...,0.067,0.949,0.945,0.984,0.028,0.068,0.934,1.000,1.000,1.0



PRIMARY: pereira3c ['c7', 'c6', 'c2'] | epoch 11 | T=2.123 | threshold=0.96 | constraints_met=True


In [13]:
# ================================================================
# CELL 12B — EVENT-LEVEL FIRE CONFIDENCE (calibration + reliability check)
# ================================================================
# The packet field means "this EVENT is a real fire", so it is calibrated per EVENT and per SIZE CLASS:
#   event score  = mean calibrated pixel probability inside the connected component   (as in CELL 13)
#   event label  = 1 if the component lies within EVENT_TOL_PX of a reference fire event
#   mapping      = isotonic regression per predicted-area class (pooled fallback when data is thin)
# Fitted on VAL-CAL (scene-disjoint from the epoch/threshold selection), reported on the held-out TEST set.
#
# IMPORTANT: the reference is the SWIR-voting label, so "confidence" = agreement with that consensus.
# Only the human-annotated scenes (CELL 20) say anything about agreement with real fires.
# ================================================================
from sklearn.isotonic import IsotonicRegression

MIN_EVENTS_PER_CLASS = 150


@torch.no_grad()
def collect_predicted_events(model, loader, threshold, temperature):
    """One row per PREDICTED event: mean calibrated prob, max raw logit, area, real(1)/false(0)."""
    model.eval()
    out = []
    for images, masks in loader:
        z = model(images.to(device, non_blocking=True)).float().cpu().numpy()[:, 0]
        probs = sigmoid_temperature_np(z, temperature)
        gts = masks.numpy()[:, 0] > 0.5
        for zi, p, g in zip(z, probs, gts):
            pred = p >= threshold
            if not pred.any():
                continue
            lab, n = ndimage.label(pred, structure=_S8)
            idx = np.arange(1, n + 1)
            gnear = ndimage.binary_dilation(g, structure=_S8, iterations=EVENT_TOL_PX) if g.any() else g
            area = np.atleast_1d(ndimage.sum(pred, lab, idx)).astype(int)
            meanp = np.atleast_1d(ndimage.mean(p, lab, idx))
            maxz = np.atleast_1d(ndimage.maximum(zi, lab, idx))
            real = np.atleast_1d(ndimage.maximum(gnear.astype(np.uint8), lab, idx)) > 0
            for a, m, mz, r in zip(area, meanp, maxz, real):
                out.append((int(a), float(m), float(mz), int(r)))
    return pd.DataFrame(out, columns=['area', 'mean_p', 'max_logit', 'real'])


def _class_of(area):
    return next(n for lo, hi, n in EVENT_AREA_CLASSES if lo <= int(area) <= hi)


def fit_event_confidence_table(ev_df, grid_lo):
    ev_df = ev_df.assign(cls=ev_df['area'].map(_class_of))
    grid = np.linspace(max(grid_lo, 0.0), 1.0, 41)

    def _fit(d):
        iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds='clip')
        iso.fit(d['mean_p'].to_numpy(), d['real'].to_numpy())
        return {'x': [float(v) for v in grid], 'y': [float(v) for v in iso.predict(grid)], 'n': int(len(d))}

    classes = {'all': _fit(ev_df)}
    for name in [n for _, _, n in EVENT_AREA_CLASSES]:
        d = ev_df[ev_df['cls'] == name]
        if len(d) >= MIN_EVENTS_PER_CLASS and 0 < d['real'].sum() < len(d):
            classes[name] = _fit(d)
    return {'method': 'isotonic regression of event-real on mean calibrated probability, per predicted-area class',
            'event_tolerance_px': EVENT_TOL_PX, 'classes': classes}


def reliability(conf, real, n_bins=10):
    conf, real = np.asarray(conf, float), np.asarray(real, float)
    edges = np.linspace(0, 1, n_bins + 1)
    b = np.clip(np.digitize(conf, edges[1:-1]), 0, n_bins - 1)
    rows, ece = [], 0.0
    for k in range(n_bins):
        m = b == k
        if m.any():
            rows.append({'bin': f'{edges[k]:.1f}-{edges[k + 1]:.1f}', 'events': int(m.sum()),
                         'mean_confidence': float(conf[m].mean()), 'observed_real_fraction': float(real[m].mean())})
            ece += m.mean() * abs(conf[m].mean() - real[m].mean())
    return pd.DataFrame(rows), float(ece)


# ---------------- run on the PRIMARY model ----------------
_cfg = PRIMARY_CONFIG
_T, _thr = OPERATING[_cfg]['temperature'], OPERATING[_cfg]['threshold']
_m, _ = load_model_from_checkpoint(TRAINED[_cfg]['ckpt'])

ev_cal = collect_predicted_events(_m, make_frame_loader(VAL_CAL_DF, PRIMARY_NAMES), _thr, _T)
ev_test = collect_predicted_events(_m, make_frame_loader(test_df, PRIMARY_NAMES), _thr, _T)
print(f'predicted events: VAL-CAL {len(ev_cal):,} (real {int(ev_cal.real.sum()):,}) | '
      f'TEST {len(ev_test):,} (real {int(ev_test.real.sum()):,})')

table = fit_event_confidence_table(ev_cal, grid_lo=_thr)
EVENT_CONF_TABLE = table                      # installs the mapping used by event_confidence() / CELL 13 patch

ev_test = ev_test.assign(
    conf_raw=ev_test['mean_p'],
    conf_cal=[apply_event_confidence(m, a) for m, a in zip(ev_test['mean_p'], ev_test['area'])],
    cls=ev_test['area'].map(_class_of))

rel_raw, ece_raw = reliability(ev_test['conf_raw'], ev_test['real'])
rel_cal, ece_cal = reliability(ev_test['conf_cal'], ev_test['real'])
print(f'\nEVENT RELIABILITY ON TEST  |  ECE raw mean-probability = {ece_raw:.4f}  ->  calibrated = {ece_cal:.4f}')
print('(ECE = average |confidence - observed real fraction|; lower is better)')
display(rel_cal.round(3))

by_cls = []
for name in [n for _, _, n in EVENT_AREA_CLASSES]:
    d = ev_test[ev_test['cls'] == name]
    if len(d):
        by_cls.append({'predicted_event_size': name, 'events': len(d), 'observed_real_fraction': d['real'].mean(),
                       'mean_conf_raw': d['conf_raw'].mean(), 'mean_conf_calibrated': d['conf_cal'].mean()})
by_cls = pd.DataFrame(by_cls)
display(by_cls.round(3))

rel_cal.to_csv(OUTPUT_DIR / 'event_reliability_test_v7.csv', index=False)
by_cls.to_csv(OUTPUT_DIR / 'event_confidence_by_size_test_v7.csv', index=False)
(OUTPUT_DIR / 'event_confidence_calibration_v7.json').write_text(json.dumps({
    'config': _cfg, 'raw_threshold_on_calibrated_probability': _thr, 'temperature': _T,
    'table': table, 'test_ece_raw': ece_raw, 'test_ece_calibrated': ece_cal,
    'meaning': 'P(event agrees with the SWIR-voting reference within %d px). Not verified against ground truth; '
               'see CELL 20 for human-annotated scenes.' % EVENT_TOL_PX}, indent=2))

del _m
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('\nTo transmit this confidence in the packet, make ONE edit in CELL 13 (function extract_events):\n'
      '   replace   confidence = float(np.mean(prob[component])) if component.any() else float(threshold)\n'
      '   with      confidence = event_confidence(prob, component, float(threshold))')


predicted events: VAL-CAL 3,214 (real 3,161) | TEST 8,365 (real 8,228)

EVENT RELIABILITY ON TEST  |  ECE raw mean-probability = 0.0150  ->  calibrated = 0.0020
(ECE = average |confidence - observed real fraction|; lower is better)


,bin,events,mean_confidence,observed_real_fraction
0,0.0-0.1,6,0.016,0.667
1,0.1-0.2,1,0.192,1.000
2,0.8-0.9,1,0.868,1.000
3,0.9-1.0,8357,0.982,0.984


,predicted_event_size,events,observed_real_fraction,mean_conf_raw,mean_conf_calibrated
0,1-5 px,6799,0.980,0.999,0.981
1,6-25 px,1367,0.999,0.999,0.984
2,26-100 px,183,1.000,0.999,0.976
3,>100 px,16,1.000,0.999,0.967



To transmit this confidence in the packet, make ONE edit in CELL 13 (function extract_events):
   replace   confidence = float(np.mean(prob[component])) if component.any() else float(threshold)
   with      confidence = event_confidence(prob, component, float(threshold))


In [14]:
# ================================================================
# CELL 13 — ONBOARD PIPELINE COMPONENTS
#   Gate 1 -> preprocessing -> CNN -> calibrated probabilities -> events
#   -> Gate 2 -> <=200-byte packet / heartbeat
# ================================================================
# The packet is an information product, not a compressed image.
# Pereira-3C (C7+C6+C2) is the current SWIR3 payload proxy.
# B10 is retained as a thermal control experiment.
# Landsat is a proxy dataset, so the future payload still needs its own sensor-specific calibration.

FRAME_SIZE = 256
MAX_EVENTS = 6
MAX_PACKET_BYTES = 200
MIN_EVENT_PIXELS = 1

# ---------------- packet / health flags ----------------
FLAG_INPUT_REJECTED = 1
FLAG_OUTPUT_REJECTED = 2
FLAG_TRANSIENTS_SCRUBBED = 4
FLAG_SATURATED_INPUT = 8
FLAG_GATE2_WARN = 16
FLAG_HEARTBEAT = 32
FLAG_TIME_UNAVAILABLE = 64
FLAG_REFERENCE_ONLY = 128

# ---------------- event flags ----------------
EVENT_FLAG_NO_GEO = 1
EVENT_FLAG_GEO_UNCERT_UNKNOWN = 2
EVENT_FLAG_SPECTRAL_METRIC = 4
EVENT_FLAG_THERMAL_METRIC = 8

# Header = magic, version, model_id, frame_id, acquisition_timestamp_ms, n_events, flags = 17 B
HEADER_FMT = '<BBBIQBB'
# Event = lat_e7, lon_e7, x, y, w, h, area_ha_x10, confidence_u16,
#         metric_u16, gsd_m_u16, geo_uncertainty_m_u16, flags = 27 B
EVENT_FMT = '<iiHHHHHHHHHB'
PKT_MAGIC, PKT_VERSION = 0xF1, 3

HEADER_BYTES = struct.calcsize(HEADER_FMT)
EVENT_BYTES = struct.calcsize(EVENT_FMT)
MAX_ALERT_PACKET_BYTES = HEADER_BYTES + MAX_EVENTS * EVENT_BYTES + 4
MAX_EMPTY_PACKET_BYTES = HEADER_BYTES + 4
assert MAX_ALERT_PACKET_BYTES <= MAX_PACKET_BYTES
print('Packet sizes: header=', HEADER_BYTES, 'event=', EVENT_BYTES,
      'max alert=', MAX_ALERT_PACKET_BYTES, 'max empty/heartbeat=', MAX_EMPTY_PACKET_BYTES,
      'limit=', MAX_PACKET_BYTES)


def _event_metric(dn, names, component):
    """Return a non-absolute event contrast metric.

    B10: 95th-percentile brightness-temperature hotspot contrast relative to a local ring.
    Pereira-3C: 95th-percentile SWIR2 reflectance contrast relative to a local ring.
    The metric meaning is encoded by an event flag; it is never called temperature for SWIR.
    """
    component = component.astype(bool)
    dil = ndimage.binary_dilation(component, structure=_S8, iterations=5)
    ring = dil & ~component

    if 'b10' in names:
        field = dn_to_bt_kelvin(dn[names.index('b10')])
        metric = float(np.percentile(field[component], 95) - np.median(field[ring])) if ring.any() else 0.0
        return max(metric, 0.0), 100.0, EVENT_FLAG_THERMAL_METRIC

    field = dn_to_reflectance(dn[names.index('c7')])
    metric = float(np.percentile(field[component], 95) - np.median(field[ring])) if ring.any() else 0.0
    return max(metric, 0.0), 10000.0, EVENT_FLAG_SPECTRAL_METRIC


def extract_events(prob, dn, names, threshold, min_pixels=MIN_EVENT_PIXELS, geo_row=None,
                   gsd_m=GSD_M_TEST, location_uncertainty_m=LOCATION_UNCERTAINTY_UNKNOWN_M):
    """Convert calibrated probability map into <=6 compact fire events."""
    mask = prob >= threshold
    labels, n = ndimage.label(mask, structure=_S8)
    if n == 0:
        return []

    idx = np.arange(1, n + 1)
    areas = np.atleast_1d(ndimage.sum(mask, labels, idx)).astype(int)
    objs = ndimage.find_objects(labels)
    events = []

    for k in np.flatnonzero(areas >= min_pixels):
        y0, y1 = objs[k][0].start, objs[k][0].stop
        x0, x1 = objs[k][1].start, objs[k][1].stop
        component = labels == (k + 1)

        # Confidence is intentionally NOT max(probability): that saturates at 1.0.
        # Mean event probability after scalar temperature calibration preserves more information.
        confidence = event_confidence(prob, component, float(threshold))   # CELL 9 hook (CELL 12B installs the event calibration table)
        metric, scale, metric_flag = _event_metric(dn, names, component)
        cy, cx = ndimage.center_of_mass(component)
        lat, lon = pixel_to_latlon(geo_row, cx, cy)
        ok_geo = np.isfinite(lat) and np.isfinite(lon)

        if ok_geo:
            lat_e7, lon_e7 = int(round(lat * 1e7)), int(round(lon * 1e7))
        else:
            lat_e7 = lon_e7 = 0

        geo_unc = int(np.clip(round(float(location_uncertainty_m)), 0, 65535))
        event_flags = metric_flag
        if not ok_geo:
            event_flags |= EVENT_FLAG_NO_GEO
        if geo_unc >= LOCATION_UNCERTAINTY_UNKNOWN_M:
            event_flags |= EVENT_FLAG_GEO_UNCERT_UNKNOWN

        # 0.1 hectare units: area_m2 / 1000 = area in 0.1-ha units.
        area_ha_x10 = int(np.clip(round(float(areas[k]) * float(gsd_m) ** 2 / 1000.0), 0, 65535))
        metric_u16 = int(np.clip(round(metric * scale), 0, 65535))
        gsd_u16 = int(np.clip(round(float(gsd_m)), 0, 65535))

        events.append({
            'lat_e7': lat_e7,
            'lon_e7': lon_e7,
            'x': int(x0), 'y': int(y0), 'w': int(x1 - x0), 'h': int(y1 - y0),
            'area_ha_x10': area_ha_x10,
            'confidence_u16': int(np.clip(round(confidence * 65535), 0, 65535)),
            'metric_u16': metric_u16,
            'gsd_m_u16': gsd_u16,
            'geo_uncertainty_m_u16': geo_unc,
            'flags': int(event_flags),
        })

    # Highest confidence first; area breaks ties.
    events.sort(key=lambda e: (e['confidence_u16'], e['area_ha_x10']), reverse=True)
    return events[:MAX_EVENTS]


def apply_min_size(mask, min_pixels):
    if min_pixels <= 1 or not mask.any():
        return mask
    labels, n = ndimage.label(mask, structure=_S8)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_pixels
    keep[0] = False
    return keep[labels]


# ---------------- Gate 2 ----------------
class Gate2:
    def __init__(self, threshold, max_fire_frac=0.05):
        self.threshold, self.max_fire_frac = threshold, max_fire_frac

    def validate(self, prob, events):
        if not np.isfinite(prob).all() or prob.min() < -1e-3 or prob.max() > 1 + 1e-3:
            return {'status': 'REJECT', 'reasons': ['probability map invalid'], 'flags': FLAG_OUTPUT_REJECTED}
        for e in events:
            if e['x'] + e['w'] > FRAME_SIZE or e['y'] + e['h'] > FRAME_SIZE or e['area_ha_x10'] <= 0:
                return {'status': 'REJECT', 'reasons': ['event geometry/area invalid'], 'flags': FLAG_OUTPUT_REJECTED}
        if float((prob >= self.threshold).mean()) > self.max_fire_frac:
            return {'status': 'WARN', 'reasons': ['implausibly large detected extent'], 'flags': FLAG_GATE2_WARN}
        return {'status': 'OK', 'reasons': [], 'flags': 0}


# ---------------- packet ----------------
def encode_packet(frame_id, model_id, acquisition_timestamp_ms, events, flags=0):
    pkt = bytearray(struct.pack(
        HEADER_FMT, PKT_MAGIC, PKT_VERSION, int(model_id) & 0xFF,
        int(frame_id) & 0xFFFFFFFF, int(acquisition_timestamp_ms) & 0xFFFFFFFFFFFFFFFF,
        len(events), int(flags) & 0xFF
    ))
    for e in events:
        pkt.extend(struct.pack(
            EVENT_FMT,
            e['lat_e7'], e['lon_e7'], e['x'], e['y'], e['w'], e['h'],
            e['area_ha_x10'], e['confidence_u16'], e['metric_u16'],
            e['gsd_m_u16'], e['geo_uncertainty_m_u16'], e['flags']
        ))
    pkt.extend(struct.pack('<I', zlib.crc32(bytes(pkt)) & 0xFFFFFFFF))
    assert len(pkt) <= MAX_PACKET_BYTES
    return bytes(pkt)


def decode_packet(data):
    if len(data) < HEADER_BYTES + 4:
        raise ValueError('packet too short')
    body, crc = data[:-4], struct.unpack('<I', data[-4:])[0]
    if zlib.crc32(body) & 0xFFFFFFFF != crc:
        raise ValueError('CRC mismatch')
    magic, ver, model_id, frame_id, acq_ts, n, flags = struct.unpack_from(HEADER_FMT, body, 0)
    if magic != PKT_MAGIC or ver != PKT_VERSION:
        raise ValueError('unsupported packet header')
    if len(body) != HEADER_BYTES + n * EVENT_BYTES:
        raise ValueError('packet length/event count mismatch')

    off, events = HEADER_BYTES, []
    names = ['lat_e7', 'lon_e7', 'x', 'y', 'w', 'h', 'area_ha_x10', 'confidence_u16',
             'metric_u16', 'gsd_m_u16', 'geo_uncertainty_m_u16', 'flags']
    for _ in range(n):
        v = struct.unpack_from(EVENT_FMT, body, off)
        off += EVENT_BYTES
        events.append(dict(zip(names, v)))

    return {
        'magic': magic, 'version': ver, 'model_id': model_id, 'frame_id': frame_id,
        'acquisition_timestamp_ms': acq_ts, 'n_events': n, 'flags': flags, 'events': events,
    }


def urgency_score(events, names):
    """Priority from confidence + contrast metric + event area; confidence is no longer max-pixel saturated."""
    best = 0.0
    for e in events:
        conf = e['confidence_u16'] / 65535.0
        if e['flags'] & EVENT_FLAG_THERMAL_METRIC:
            delta_k = e['metric_u16'] / 100.0
            metric = float(np.clip(delta_k / 40.0, 0.0, 1.0))
        else:
            delta_rho = e['metric_u16'] / 10000.0
            metric = float(np.clip(delta_rho / 0.5, 0.0, 1.0))
        area_ha = e['area_ha_x10'] / 10.0
        size = min(math.log1p(area_ha) / math.log1p(100.0), 1.0)
        best = max(best, 0.5 * conf + 0.3 * metric + 0.2 * size)
    return best


# ---------------- pipeline ----------------
class FirePipeline:
    """Single-frame pipeline. Acquisition time and GSD are supplied by the sensor/platform layer."""

    def __init__(self, session, input_name, names, threshold, temperature=1.0,
                 model_id=None, reference_only=False, gate1_scrub=False,
                 min_event_pixels=MIN_EVENT_PIXELS, heartbeat_every_n_frames=30):
        self.session, self.input_name, self.names = session, input_name, list(names)
        self.threshold = float(threshold)
        self.temperature = float(temperature)
        self.model_id = int(model_id if model_id is not None else MODEL_IDS[PRIMARY_CONFIG])
        self.reference_only = bool(reference_only)
        self.min_event_pixels = int(min_event_pixels)
        self.heartbeat_every_n_frames = int(heartbeat_every_n_frames)
        self.gate1 = Gate1(names, scrub=gate1_scrub)
        self.gate2 = Gate2(self.threshold)
        self.frames_processed = 0

    def process(self, dn, frame_id, acquisition_timestamp_ms=0, geo_row=None,
                gsd_m=GSD_M_TEST, location_uncertainty_m=LOCATION_UNCERTAINTY_UNKNOWN_M,
                keep_prob=False):
        self.frames_processed += 1
        T = {}
        base_flags = FLAG_REFERENCE_ONLY if self.reference_only else 0
        if int(acquisition_timestamp_ms) <= 0:
            base_flags |= FLAG_TIME_UNAVAILABLE
        if float(location_uncertainty_m) >= LOCATION_UNCERTAINTY_UNKNOWN_M:
            # This is retained at event level so each event says that the uncertainty is unknown.
            pass

        t0 = time.perf_counter()
        g1 = self.gate1.validate(dn)
        T['gate1_ms'] = (time.perf_counter() - t0) * 1e3
        if g1['status'] == 'REJECT':
            pkt = encode_packet(frame_id, self.model_id, acquisition_timestamp_ms, [],
                                flags=base_flags | g1['flags'])
            T['encode_ms'] = 0.0
            T['total_ms'] = sum(T.values())
            return {'status': 'GATE1_REJECT', 'packet': pkt, 'events': [], 'urgency': 0.6,
                    'timings': T, 'prob': None, 'n_transients': g1['n_transients']}

        t0 = time.perf_counter()
        x = preprocess_dn(g1['dn'], self.names)[None]
        T['preprocess_ms'] = (time.perf_counter() - t0) * 1e3

        t0 = time.perf_counter()
        logits = self.session.run(None, {self.input_name: x})[0][0, 0]
        prob = sigmoid_temperature_np(logits, self.temperature)
        T['infer_ms'] = (time.perf_counter() - t0) * 1e3

        t0 = time.perf_counter()
        events = extract_events(prob, g1['dn'], self.names, self.threshold,
                                self.min_event_pixels, geo_row, gsd_m, location_uncertainty_m)
        T['decide_ms'] = (time.perf_counter() - t0) * 1e3

        t0 = time.perf_counter()
        g2 = self.gate2.validate(prob, events)
        T['gate2_ms'] = (time.perf_counter() - t0) * 1e3

        flags = base_flags | g1['flags'] | g2['flags']
        t0 = time.perf_counter()
        if g2['status'] == 'REJECT':
            pkt = encode_packet(frame_id, self.model_id, acquisition_timestamp_ms, [], flags=flags)
            status, urg, events = 'GATE2_REJECT', 0.6, []
        elif events:
            pkt = encode_packet(frame_id, self.model_id, acquisition_timestamp_ms, events, flags=flags)
            status, urg = 'ALERT', urgency_score(events, self.names)
        else:
            heartbeat_due = (self.frames_processed == 1 or
                             (self.heartbeat_every_n_frames > 0 and
                              self.frames_processed % self.heartbeat_every_n_frames == 0))
            if heartbeat_due:
                pkt = encode_packet(frame_id, self.model_id, acquisition_timestamp_ms, [],
                                    flags=flags | FLAG_HEARTBEAT)
                status, urg = 'HEARTBEAT_NO_FIRE', 0.0
            else:
                pkt, status, urg = b'', 'NO_FIRE', 0.0
        T['encode_ms'] = (time.perf_counter() - t0) * 1e3
        T['total_ms'] = sum(v for k, v in T.items() if k != 'total_ms')
        return {'status': status, 'packet': pkt, 'events': events, 'urgency': urg, 'timings': T,
                'prob': prob.astype(np.float16) if keep_prob else None, 'n_transients': g1['n_transients']}


print('Pipeline components defined.')


Packet sizes: header= 17 event= 27 max alert= 183 max empty/heartbeat= 21 limit= 200
Pipeline components defined.


In [15]:
# ================================================================
# CELL 13B — GATE 1 (input validation + transient detection)   [run right after CELL 13]
# ================================================================
# NOTE: the uploaded notebook references Gate1 but never defines it (the cell was lost during the
# edits). This is a reconstruction that matches how CELL 13 / 14 / 17 call it:
#     Gate1(names, scrub=False).validate(dn) -> {'status', 'reasons', 'flags', 'dn', 'n_transients'}
# If you still have your ORIGINAL Gate1 cell, keep that one and skip this cell.
#
# Transient = an ISOLATED saturated pixel (value == 65535 with at most one saturated neighbour in 3x3).
# scrub=False (default): count/flag only.  scrub=True: replace by the 3x3 median.
# WARNING: an intense 1-pixel fire can saturate SWIR2 -> scrub=True can delete a real 1-px fire.
# ================================================================
from scipy.ndimage import median_filter     # (ndimage is imported in CELL 1)

SAT_DN = 65535


class Gate1:
    def __init__(self, names, scrub=False, max_saturated_frac=0.02, min_valid_frac=0.05,
                 reject_saturated_frac=0.50, max_dead_col_frac=0.20):
        self.names = list(names)
        self.scrub = bool(scrub)
        self.max_saturated_frac = max_saturated_frac
        self.min_valid_frac = min_valid_frac
        self.reject_saturated_frac = reject_saturated_frac
        self.max_dead_col_frac = max_dead_col_frac

    def _reject(self, dn, reason):
        return {'status': 'REJECT', 'reasons': [reason], 'flags': FLAG_INPUT_REJECTED, 'dn': dn, 'n_transients': 0}

    def validate(self, dn):
        dn = np.asarray(dn)
        C = len(self.names)
        if dn.ndim != 3 or dn.shape != (C, FRAME_SIZE, FRAME_SIZE):
            return self._reject(dn, f'bad shape {dn.shape}, expected {(C, FRAME_SIZE, FRAME_SIZE)}')
        if np.issubdtype(dn.dtype, np.floating):
            if not np.isfinite(dn).all():
                return self._reject(dn, 'non-finite input values')
            dn = np.clip(dn, 0, SAT_DN)
        dn = dn.astype(np.uint16, copy=False)

        valid = (dn > 0).any(0)                      # zero-padded / no-data border is all-zero in every band
        if valid.mean() < self.min_valid_frac:
            return self._reject(dn, f'valid area {valid.mean():.1%} < {self.min_valid_frac:.0%}')

        reasons, flags = [], 0
        for k, nm in enumerate(self.names):
            band = dn[k]
            if band[valid].min() == band[valid].max():
                return self._reject(dn, f'band {nm} is constant (dead channel)')
            sat = float((band >= SAT_DN).mean())
            if sat >= self.reject_saturated_frac:
                return self._reject(dn, f'band {nm} saturated on {sat:.0%} of pixels')
            if sat > self.max_saturated_frac:
                flags |= FLAG_SATURATED_INPUT
                reasons.append(f'band {nm}: {sat:.1%} saturated pixels')
            if C > 1:
                others_have_data = (np.delete(dn, k, axis=0) > 0).any(axis=(0, 1))   # per column
                dead = (band == 0).all(axis=0) & others_have_data
                if dead.mean() > self.max_dead_col_frac:
                    return self._reject(dn, f'band {nm}: {dead.mean():.0%} dead columns')
                if dead.any():
                    reasons.append(f'band {nm}: {int(dead.sum())} dead columns')

        n_trans = 0
        out = dn
        kernel = np.ones((3, 3), np.int32)
        for k in range(C):
            band = dn[k]
            sat = band >= SAT_DN
            if not sat.any():
                continue
            # isolated = saturated pixel with at most ONE saturated neighbour in its 3x3 window
            # (real saturated clouds/cores are larger blobs and are left alone)
            n_sat_nb = ndimage.convolve(sat.astype(np.int32), kernel, mode='constant') - 1
            spike = sat & (n_sat_nb <= 1)
            n = int(spike.sum())
            if n:
                n_trans += n
                if self.scrub:
                    if out is dn:
                        out = dn.copy()
                    med = median_filter(band, size=3, mode='nearest')
                    out[k][spike] = med[spike]
        if n_trans:
            reasons.append(f'{n_trans} isolated saturated pixel(s)' + (' scrubbed' if self.scrub else ' (not scrubbed)'))
            if self.scrub:
                flags |= FLAG_TRANSIENTS_SCRUBBED
        return {'status': 'OK', 'reasons': reasons, 'flags': flags, 'dn': out, 'n_transients': n_trans}


In [16]:
# ================================================================
# CELL 14 — RAW-INPUT ROBUSTNESS (sensor degradations) + Gate 1 TRANSIENT SCRUBBING
# ================================================================
# Degradations are applied to the RAW uint16 DN, before preprocessing, like real sensor effects:
#   noise (Gaussian, DN), quantization (fewer bits), transients (radiation hits / hot pixels:
#   single-pixel saturated spikes in one band), dead columns (one band, whole column = 0).
# Question answered: how much accuracy does the model lose, and does Gate 1 scrubbing help
# against transients (at the cost of some real 1-pixel fires)?

rng_sel = np.random.default_rng(SEED)
rob_rows = rng_sel.choice(test_df['cache_idx'].to_numpy(), size=min(ROBUSTNESS_N, len(test_df)), replace=False)
rob_slots = channel_slots(PRIMARY_NAMES)

NOISE_SIGMA_DN = {1: 100, 2: 300, 3: 800}       # reflective bands (0.002 / 0.006 / 0.016 reflectance)
NOISE_SIGMA_B10_DN = {1: 50, 2: 150, 3: 400}
QUANT_SHIFT = {1: 4, 2: 6, 3: 8}                # 12 / 10 / 8 effective bits
N_TRANSIENTS = {1: 5, 2: 20, 3: 80}
N_DEADCOLS = {1: 1, 2: 3, 3: 8}


def degrade(dn, kind, level, rng, names):
    x = dn.astype(np.float32)
    C = x.shape[0]
    if kind == 'noise':
        for k, name in enumerate(names):
            sigma = (NOISE_SIGMA_B10_DN if name == 'b10' else NOISE_SIGMA_DN)[level]
            x[k] += rng.normal(0.0, sigma, x[k].shape)
    elif kind == 'quantization':
        step = float(2 ** QUANT_SHIFT[level])
        x = np.floor(x / step) * step
    elif kind == 'transients':
        for _ in range(N_TRANSIENTS[level]):
            b, yy, xx = int(rng.integers(C)), int(rng.integers(FRAME_SIZE)), int(rng.integers(FRAME_SIZE))
            x[b, yy, xx] = 65535.0
    elif kind == 'dead_columns':
        for _ in range(N_DEADCOLS[level]):
            x[int(rng.integers(C)), :, int(rng.integers(FRAME_SIZE))] = 0.0
    return np.clip(x, 0, 65535).astype(np.uint16)


def run_scenario(kind, level, scrub=False, batch=16):
    rng = np.random.default_rng(SEED + 1)                  # same noise realisation in every scenario
    gate = Gate1(PRIMARY_NAMES, scrub=True) if scrub else None
    tp = fp = fn = 0
    n_fire = n_nofire = detected_fire = false_alarm = 0
    ev_total = ev_found = ev_big_total = ev_big_found = pred_events = false_events = 0
    primary_model.eval()
    for s in range(0, len(rob_rows), batch):
        chunk = rob_rows[s:s + batch]
        frames = []
        for i in chunk:
            dn = np.asarray(BANDS_MM[int(i)][rob_slots])
            if kind != 'clean':
                dn = degrade(dn, kind, level, rng, PRIMARY_NAMES)
            if gate is not None:
                dn = gate.validate(dn)['dn']
            frames.append(preprocess_dn(dn, PRIMARY_NAMES))
        with torch.no_grad():
            prob = sigmoid_temperature_np(primary_model(torch.from_numpy(np.stack(frames)).to(device)).float().cpu().numpy(), PRIMARY_TEMPERATURE)[:, 0]
        for p, i in zip(prob, chunk):
            g = np.asarray(MASKS_MM[int(i)]) > 0
            pred = p >= PRIMARY_THRESHOLD
            tp += int((pred & g).sum()); fp += int((pred & ~g).sum()); fn += int((~pred & g).sum())
            if g.any():
                n_fire += 1; detected_fire += int(pred.any())
            else:
                n_nofire += 1; false_alarm += int(pred.any())
            sizes, found, npd, nfa = match_events(pred, g)
            ev_total += len(sizes); ev_found += int(found.sum())
            big = sizes >= 26
            ev_big_total += int(big.sum()); ev_big_found += int(found[big].sum())
            pred_events += npd; false_events += nfa
    m = metrics_from_counts(tp, fp, fn)
    return {'scenario': kind if kind == 'clean' else f'{kind} L{level}' + (' + Gate1 scrub' if scrub else ''),
            'pixel_F1': m['f1'], 'pixel_recall': m['recall'],
            'frame_recall': detected_fire / max(n_fire, 1), 'frame_FAR': false_alarm / max(n_nofire, 1),
            'event_recall_all': ev_found / max(ev_total, 1), 'event_recall_>=26px': ev_big_found / max(ev_big_total, 1),
            'false_events_per_patch': false_events / max(len(rob_rows), 1)}


scenarios = [('clean', 0, False)]
for kind in ('noise', 'quantization', 'transients', 'dead_columns'):
    scenarios += [(kind, lvl, False) for lvl in (1, 2, 3)]
scenarios += [('transients', 2, True), ('transients', 3, True), ('clean', 0, True)]

robust_rows = [run_scenario(k, l, s) for k, l, s in tqdm(scenarios, desc='Robustness scenarios')]
robust_df = pd.DataFrame(robust_rows)
display(robust_df.round(3))
robust_df.to_csv(OUTPUT_DIR / 'robustness_primary_v7.csv', index=False)
print('Read: "transients L3" vs "transients L3 + Gate1 scrub" = benefit of Gate 1; "clean + Gate1 scrub" vs "clean" = '
      'collateral damage on real fires (isolated 1-pixel fires can look like transients).')


Robustness scenarios:   0%|          | 0/16 [00:00<?, ?it/s]

,scenario,pixel_F1,pixel_recall,frame_recall,frame_FAR,event_recall_all,event_recall_>=26px,false_events_per_patch
0,clean,0.952,0.954,0.966,0.067,0.958,1.0,0.025
1,noise L1,0.951,0.953,0.967,0.061,0.958,1.0,0.022
2,noise L2,0.945,0.950,0.970,0.078,0.956,1.0,0.040
3,noise L3,0.844,0.923,0.970,0.313,0.945,1.0,1.351
4,quantization L1,0.952,0.954,0.966,0.067,0.959,1.0,0.026
5,quantization L2,0.952,0.955,0.967,0.067,0.959,1.0,0.026
6,quantization L3,0.953,0.957,0.968,0.067,0.962,1.0,0.024
7,transients L1,0.865,0.954,0.996,0.821,0.957,1.0,1.601
8,transients L2,0.672,0.952,1.000,1.000,0.953,1.0,6.526
9,transients L3,0.351,0.949,1.000,1.000,0.952,1.0,26.577


Read: "transients L3" vs "transients L3 + Gate1 scrub" = benefit of Gate 1; "clean + Gate1 scrub" vs "clean" = collateral damage on real fires (isolated 1-pixel fires can look like transients).


In [17]:
# ================================================================
# CELL 15 — EXPORT THE PRIMARY MODEL: FP32 ONNX + STATIC INT8
# ================================================================
import onnx
from onnxsim import simplify
from onnxruntime.quantization import (CalibrationDataReader, CalibrationMethod, quantize_static,
                                      QuantFormat, QuantType)

C_IN = len(PRIMARY_NAMES)
export_model, _ck = load_model_from_checkpoint(TRAINED[PRIMARY_CONFIG]['ckpt'])
export_model = export_model.cpu().eval()
example = torch.zeros(1, C_IN, FRAME_SIZE, FRAME_SIZE, dtype=torch.float32)

onnx_raw = OUTPUT_DIR / f'unet_{PRIMARY_CONFIG}_fp32_raw.onnx'
export_kwargs = dict(input_names=['bands'], output_names=['fire_logits'], opset_version=17,
                     do_constant_folding=True)          # static batch = 1 (one frame at a time onboard)
try:
    torch.onnx.export(export_model, example, str(onnx_raw), dynamo=False, **export_kwargs)
except TypeError:                                       # older torch without the `dynamo` argument
    torch.onnx.export(export_model, example, str(onnx_raw), **export_kwargs)

simplified, ok = simplify(onnx.load(str(onnx_raw)))
assert ok, 'ONNX simplification failed'
onnx_fp32 = OUTPUT_DIR / f'unet_{PRIMARY_CONFIG}_fp32.onnx'
onnx.save(simplified, str(onnx_fp32))


class CalibrationReader(CalibrationDataReader):
    """Yields representative preprocessed validation frames for static INT8 calibration.
    VAL-CAL is scene-disjoint from epoch/threshold selection in CELL 12."""

    def __init__(self, loader, max_batches=32):
        self.items, n_fire = [], 0
        for b, (images, masks) in enumerate(loader):
            for image, mask in zip(images, masks):
                sample = image.unsqueeze(0).numpy().astype(np.float32)
                assert sample.shape == (1, C_IN, FRAME_SIZE, FRAME_SIZE), sample.shape
                self.items.append({'bands': sample})
                n_fire += int(bool(mask.any()))
            if b + 1 >= max_batches:
                break
        assert self.items
        print(f'INT8 calibration frames: {len(self.items)} ({n_fire} with fire)')
        self._it = iter(self.items)

    def get_next(self):
        return next(self._it, None)


onnx_int8 = OUTPUT_DIR / f'unet_{PRIMARY_CONFIG}_int8.onnx'
try:
    quantize_static(
        model_input=str(onnx_fp32), model_output=str(onnx_int8),
        calibration_data_reader=CalibrationReader(make_frame_loader(VAL_CAL_DF, PRIMARY_NAMES)),
        quant_format=QuantFormat.QDQ, activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
        per_channel=True, calibrate_method=CalibrationMethod.MinMax,
    )
except Exception as exc:
    print('⚠️ INT8 quantization failed -> the pipeline will use FP32.', repr(exc)[:300])
    onnx_int8 = None


def size_mb(path):
    return Path(path).stat().st_size / 1024 ** 2


print('FP32 ONNX:', round(size_mb(onnx_fp32), 3), 'MB')
if onnx_int8 is not None:
    print('INT8 ONNX:', round(size_mb(onnx_int8), 3), 'MB')
print('Parameters:', count_params(export_model), '| MACs/frame:', round(count_macs(export_model, C_IN) / 1e9, 3), 'G')
del export_model


/tmp/ipykernel_49/3142373332.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(export_model, example, str(onnx_raw), dynamo=False, **export_kwargs)


INT8 calibration frames: 512 (435 with fire)


FP32 ONNX: 1.848 MB
INT8 ONNX: 0.503 MB
Parameters: 482737 | MACs/frame: 2.319 G


In [18]:
# ================================================================
# CELL 16 — VERIFY PYTORCH vs ONNX FP32 vs ONNX INT8 (audited)
# ================================================================
# Deployment gate:
#   - no use-before-definition
#   - full verification sample for probability MAE
#   - compare pixel F1, frame recall, frame FAR and binary decision agreement
#   - INT8 is accepted only if all mission-facing tolerances pass.
import onnxruntime as ort

rng_v = np.random.default_rng(SEED + 7)
verify_rows = rng_v.choice(len(test_df), size=min(VERIFY_N, len(test_df)), replace=False)
verify_df = test_df.iloc[np.sort(verify_rows)]
verify_loader = DataLoader(
    BandDataset(verify_df, PRIMARY_NAMES),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

sessions = {
    'ONNX FP32': ort.InferenceSession(str(onnx_fp32), providers=['CPUExecutionProvider'])
}
if onnx_int8 is not None:
    sessions['ONNX INT8'] = ort.InferenceSession(
        str(onnx_int8),
        providers=['CPUExecutionProvider']
    )

def new_acc():
    return dict(tp=0, fp=0, fn=0, fire=0, nofire=0, hit=0, fa=0, alerts=0)

def add_acc(a, pred, gt):
    a['tp'] += int((pred & gt).sum())
    a['fp'] += int((pred & ~gt).sum())
    a['fn'] += int((~pred & gt).sum())
    gf = gt.reshape(len(gt), -1).any(1)
    pf = pred.reshape(len(pred), -1).any(1)
    a['fire'] += int(gf.sum())
    a['nofire'] += int((~gf).sum())
    a['hit'] += int((gf & pf).sum())
    a['fa'] += int((~gf & pf).sum())
    a['alerts'] += int(pf.sum())
    return pf

acc = {'PyTorch FP32': new_acc(), **{k: new_acc() for k in sessions}}
agree = {k: 0 for k in sessions}
mae_sum = {k: 0.0 for k in sessions}
n_frames = 0

primary_model.eval()
for images, masks in tqdm(verify_loader, desc='Verifying'):
    x = images.numpy().astype(np.float32)
    gt = masks.numpy()[:, 0] > 0.5

    with torch.no_grad():
        z_t = primary_model(images.to(device)).cpu().numpy()
    p_t = sigmoid_temperature_np(z_t, PRIMARY_TEMPERATURE)[:, 0]
    pf_t = add_acc(acc['PyTorch FP32'], p_t >= PRIMARY_THRESHOLD, gt)

    for name, sess in sessions.items():
        in_name = sess.get_inputs()[0].name
        z_o = np.concatenate(
            [sess.run(None, {in_name: x[i:i + 1]})[0] for i in range(len(x))],
            axis=0
        )
        p_o = sigmoid_temperature_np(z_o, PRIMARY_TEMPERATURE)[:, 0]
        pf_o = add_acc(acc[name], p_o >= PRIMARY_THRESHOLD, gt)
        agree[name] += int((pf_o == pf_t).sum())
        mae_sum[name] += float(np.abs(p_o - p_t).mean()) * len(x)

    n_frames += len(x)

rows = []
for name, a in acc.items():
    m = metrics_from_counts(a['tp'], a['fp'], a['fn'])
    rows.append({
        'model': name,
        'pixel_F1': m['f1'],
        'pixel_recall': m['recall'],
        'pixel_precision': m['precision'],
        'frame_recall': a['hit'] / max(a['fire'], 1),
        'frame_FAR': a['fa'] / max(a['nofire'], 1),
        'alert_decision_agreement_vs_torch': 1.0 if name == 'PyTorch FP32' else agree[name] / max(n_frames, 1),
        'calibrated_prob_MAE_vs_torch': 0.0 if name == 'PyTorch FP32' else mae_sum[name] / max(n_frames, 1),
    })

verification_df = pd.DataFrame(rows)
display(verification_df.round(4))
verification_df.to_csv(OUTPUT_DIR / 'onnx_verification_v7.csv', index=False)

INT8_OK = False
if 'ONNX INT8' in sessions:
    ref = verification_df[verification_df['model'] == 'PyTorch FP32'].iloc[0]
    q = verification_df[verification_df['model'] == 'ONNX INT8'].iloc[0]

    d_f1 = q['pixel_F1'] - ref['pixel_F1']
    d_fr = q['frame_recall'] - ref['frame_recall']
    d_far = q['frame_FAR'] - ref['frame_FAR']
    d_agree = q['alert_decision_agreement_vs_torch']
    d_mae = q['calibrated_prob_MAE_vs_torch']

    print(
        f'INT8 vs PyTorch: dF1 {d_f1:+.4f} | '
        f'dFrameRecall {d_fr:+.4f} | dFAR {d_far:+.4f} | '
        f'decision agreement {d_agree:.4%} | prob MAE {d_mae:.6f}'
    )

    INT8_OK = (
        abs(d_f1) <= 0.01
        and d_fr >= -0.01
        and d_far <= 0.01
        and d_agree >= 0.99
    )

    print(
        '✅ INT8 accepted for pipeline.'
        if INT8_OK
        else
        '⚠️ INT8 rejected -> pipeline uses FP32. Re-run quantization with a representative calibration set.'
    )

PIPELINE_MODEL = 'int8' if INT8_OK else 'fp32'
PIPELINE_ONNX = onnx_int8 if INT8_OK else onnx_fp32
print('Pipeline model:', PIPELINE_MODEL, '->', PIPELINE_ONNX.name)
print(
    'Decision calibration: T =', round(PRIMARY_TEMPERATURE, 4),
    '| calibrated threshold =', PRIMARY_THRESHOLD
)


Verifying:   0%|          | 0/94 [00:00<?, ?it/s]

,model,pixel_F1,pixel_recall,pixel_precision,frame_recall,frame_FAR,alert_decision_agreement_vs_torch,calibrated_prob_MAE_vs_torch
0,PyTorch FP32,0.9489,0.9496,0.9482,0.9615,0.0786,1.00,0.0
1,ONNX FP32,0.9489,0.9496,0.9482,0.9615,0.0786,1.00,0.0
2,ONNX INT8,0.9482,0.9490,0.9475,0.9598,0.0821,0.99,0.0


INT8 vs PyTorch: dF1 -0.0007 | dFrameRecall -0.0016 | dFAR +0.0036 | decision agreement 99.0000% | prob MAE 0.000005
✅ INT8 accepted for pipeline.
Pipeline model: int8 -> unet_pereira3c_int8.onnx
Decision calibration: T = 2.1228 | calibrated threshold = 0.96


In [19]:
# ================================================================
# CELL 17 — END-TO-END PIPELINE: raw frame -> alert / heartbeat packet
# ================================================================
# This is the first V7 cell that produces the actual mission information product.
# No synthetic processing timestamp is used anymore: acquisition time must come from
# sensor/platform metadata. When it is unavailable in Landsat, the packet explicitly flags it.
# The primary Pereira-3C configuration is the current SWIR3 payload proxy. The actual flight sensor
# must later supply its own calibrated DN range, spectral response, GSD and geometry metadata.

GATE1_SCRUB = GATE1_SCRUB_DEFAULT
REFERENCE_ONLY = PRIMARY_CONFIG != FLIGHT_CONFIG

so = ort.SessionOptions()
so.intra_op_num_threads = 1
so.inter_op_num_threads = 1
pipe_session = ort.InferenceSession(str(PIPELINE_ONNX), so, providers=['CPUExecutionProvider'])
pipe = FirePipeline(
    pipe_session,
    pipe_session.get_inputs()[0].name,
    PRIMARY_NAMES,
    PRIMARY_THRESHOLD,
    temperature=PRIMARY_TEMPERATURE,
    model_id=MODEL_IDS[PRIMARY_CONFIG],
    reference_only=REFERENCE_ONLY,
    gate1_scrub=GATE1_SCRUB,
    min_event_pixels=MIN_EVENT_PIXELS,
    heartbeat_every_n_frames=HEARTBEAT_EVERY_N_FRAMES,
)

rng_e = np.random.default_rng(SEED + 11)
pipe_rows = np.sort(rng_e.choice(test_df['cache_idx'].to_numpy(), size=min(PIPELINE_N, len(test_df)), replace=False))
slots = channel_slots(PRIMARY_NAMES)


def acquisition_timestamp_for_row(i):
    """Return (timestamp_ms, unavailable).

    V7 no longer invents a processing timestamp. A real sensor adapter should populate
    df_all['acq_timestamp_ms']; until then, zero + FLAG_TIME_UNAVAILABLE is intentional.
    """
    if 'acq_timestamp_ms' in df_all.columns:
        v = df_all.iloc[int(i)]['acq_timestamp_ms']
        if pd.notna(v) and int(v) > 0:
            return int(v), False
    return 0, True


frames_out, probs = [], []
first_alert_packet = None
first_alert_decoded = None
for k, i in enumerate(tqdm(pipe_rows, desc='Pipeline')):
    dn = np.asarray(BANDS_MM[int(i)][slots])
    acq_ts, time_missing = acquisition_timestamp_for_row(i)
    res = pipe.process(
        dn,
        frame_id=int(i),
        acquisition_timestamp_ms=acq_ts,
        geo_row=GEO[int(i)],
        gsd_m=GSD_M_TEST,
        location_uncertainty_m=LOCATION_UNCERTAINTY_UNKNOWN_M,
        keep_prob=True,
    )
    dec = decode_packet(res['packet']) if res['packet'] else None
    if dec is not None:
        assert dec['frame_id'] == int(i)
        assert dec['model_id'] == MODEL_IDS[PRIMARY_CONFIG]
        assert dec['n_events'] == len(res['events'])
        if first_alert_packet is None and res['status'] == 'ALERT':
            first_alert_packet = res['packet']
            first_alert_decoded = dec

    frames_out.append({
        'cache_idx': int(i), 'status': res['status'], 'nbytes': len(res['packet']),
        'urgency': res['urgency'], 'n_events': len(res['events']),
        'n_transients': res['n_transients'], 'acquisition_time_available': not time_missing,
        'packet_hex': res['packet'].hex() if res['packet'] else '',
        **res['timings']
    })
    probs.append(res['prob'])

pipe_df = pd.DataFrame(frames_out)

# ---------------- data reduction ----------------
raw_per_frame = len(PRIMARY_NAMES) * FRAME_SIZE * FRAME_SIZE * 2
raw_10band = 10 * FRAME_SIZE * FRAME_SIZE * 2
n = len(pipe_df)
sent_frames = pipe_df[pipe_df['nbytes'] > 0]
total_sent_bytes = int(pipe_df['nbytes'].sum())
alert_frames = pipe_df[pipe_df['status'] == 'ALERT']
heartbeat_frames = pipe_df[pipe_df['status'] == 'HEARTBEAT_NO_FIRE']
reduction = pd.DataFrame([{
    'frames': n,
    'status_ALERT': int((pipe_df.status == 'ALERT').sum()),
    'status_HEARTBEAT_NO_FIRE': int((pipe_df.status == 'HEARTBEAT_NO_FIRE').sum()),
    'status_NO_FIRE (no packet)': int((pipe_df.status == 'NO_FIRE').sum()),
    'status_REJECT (health packet)': int(pipe_df.status.str.contains('REJECT').sum()),
    'raw_bytes_per_frame (selected bands)': raw_per_frame,
    'raw_bytes_per_frame (all 10 bands)': raw_10band,
    'mean_alert_packet_bytes': float(alert_frames['nbytes'].mean()) if len(alert_frames) else 0.0,
    'mean_all_sent_packet_bytes': float(sent_frames['nbytes'].mean()) if len(sent_frames) else 0.0,
    'max_packet_bytes': int(pipe_df['nbytes'].max()),
    'total_raw_MB': n * raw_per_frame / 1e6,
    'total_sent_KB': total_sent_bytes / 1e3,
    'overall_reduction_x_including_heartbeats': (n * raw_per_frame) / max(total_sent_bytes, 1),
    'per_alert_reduction_x': raw_per_frame / max(float(alert_frames['nbytes'].mean()), 1.0) if len(alert_frames) else np.nan,
}]).T.rename(columns={0: 'value'})
display(reduction)

# ---------------- pipeline detection quality ----------------
def pipeline_event_table(min_pixels):
    ev = {name: [0, 0] for _, _, name in EVENT_BUCKETS}
    n_pred = n_false = n_fire_frames = n_hit_frames = n_nofire = n_fa = 0
    for i, p in zip(pipe_rows, probs):
        g = np.asarray(MASKS_MM[int(i)]) > 0
        pred = (apply_min_size(p.astype(np.float32) >= PRIMARY_THRESHOLD, min_pixels)
                if p is not None else np.zeros_like(g))
        if g.any():
            n_fire_frames += 1
            n_hit_frames += int(pred.any())
        else:
            n_nofire += 1
            n_fa += int(pred.any())
        sizes, found, npd, nfa = match_events(pred, g)
        for sz, f in zip(sizes, found):
            for lo, hi, name in EVENT_BUCKETS:
                if lo <= sz <= hi:
                    ev[name][0] += 1
                    ev[name][1] += int(f)
        n_pred += npd
        n_false += nfa
    row = {'min_event_pixels': min_pixels, 'frame_recall': n_hit_frames / max(n_fire_frames, 1),
           'frame_FAR': n_fa / max(n_nofire, 1), 'event_precision': 1 - n_false / max(n_pred, 1),
           'false_events_per_patch': n_false / max(len(pipe_rows), 1)}
    for name, (e, f) in ev.items():
        row[f'event_recall {name}'] = f / max(e, 1)
    return row


pipeline_quality = pd.DataFrame([pipeline_event_table(k) for k in (1, 2, 3)])
display(pipeline_quality.round(3))
pipeline_quality.to_csv(OUTPUT_DIR / 'pipeline_event_quality_v7.csv', index=False)

# ---------------- latency per stage (host CPU) ----------------
stage_cols = ['gate1_ms', 'preprocess_ms', 'infer_ms', 'decide_ms', 'gate2_ms', 'encode_ms', 'total_ms']
latency = pd.DataFrame({c: {'mean': pipe_df[c].mean(), 'p50': pipe_df[c].quantile(.5),
                            'p95': pipe_df[c].quantile(.95), 'p99': pipe_df[c].quantile(.99)}
                        for c in stage_cols if c in pipe_df}).T
display(latency.round(2))
latency.to_csv(OUTPUT_DIR / 'pipeline_latency_host_v7.csv')
print('Host CPU latency only. Real CM4 latency/energy must come from CELL 19 benchmark bundle.')

pipeline_summary = {
    'frames': n,
    'model': PIPELINE_MODEL,
    'config': PRIMARY_CONFIG,
    'bands': PRIMARY_NAMES,
    'reference_only': REFERENCE_ONLY,
    'temperature': PRIMARY_TEMPERATURE,
    'benchmark_threshold': PRIMARY_THRESHOLD,
    'threshold_constraints_met_on_validation': bool(OPERATING[PRIMARY_CONFIG]['constraints_met']),
    'heartbeat_every_n_frames': HEARTBEAT_EVERY_N_FRAMES,
    'acquisition_timestamp_available': bool(pipe_df['acquisition_time_available'].all()) if len(pipe_df) else False,
    'raw_bytes_per_frame_selected_bands': raw_per_frame,
    'max_packet_bytes': int(pipe_df['nbytes'].max()),
    'overall_reduction_x_including_heartbeats': float(reduction.loc['overall_reduction_x_including_heartbeats', 'value']),
    'latency_total_p50_ms': float(latency.loc['total_ms', 'p50']),
    'latency_total_p95_ms': float(latency.loc['total_ms', 'p95']),
}

# ---------------- save a real packet artifact for inspection ----------------
if first_alert_packet is not None:
    (OUTPUT_DIR / 'sample_alert_packet.bin').write_bytes(first_alert_packet)
    (OUTPUT_DIR / 'sample_alert_packet.json').write_text(json.dumps(first_alert_decoded, indent=2))
elif len(sent_frames):
    r0 = sent_frames.iloc[0]
    (OUTPUT_DIR / 'sample_packet.bin').write_bytes(bytes.fromhex(r0['packet_hex']))

# Alerts/health packets handed to CELL 18. NO_FIRE frames are intentionally absent,
# but periodic heartbeats and validation-failure packets remain visible.
ALERTS = [{'frame': int(r.cache_idx), 'urgency': float(r.urgency), 'payload': bytes.fromhex(r.packet_hex)}
          for r in pipe_df.itertuples() if r.nbytes > 0]
print(f'{len(ALERTS)} packets queued for the next stage, {sum(len(a["payload"]) for a in ALERTS) / 1e3:.1f} KB total.')
if first_alert_decoded:
    print('Example decoded alert packet:')
    print(json.dumps(first_alert_decoded, indent=2))
else:
    print('No ALERT packet occurred in the sampled run; a health/heartbeat packet was saved instead.')

Pipeline:   0%|          | 0/1500 [00:00<?, ?it/s]

,value
frames,1.500000e+03
status_ALERT,1.233000e+03
status_HEARTBEAT_NO_FIRE,5.000000e+00
status_NO_FIRE (no packet),2.610000e+02
status_REJECT (health packet),1.000000e+00
raw_bytes_per_frame (selected bands),3.932160e+05
raw_bytes_per_frame (all 10 bands),1.310720e+06
mean_alert_packet_bytes,7.662044e+01
mean_all_sent_packet_bytes,7.635109e+01
max_packet_bytes,1.830000e+02


,min_event_pixels,frame_recall,frame_FAR,event_precision,false_events_per_patch,event_recall 1-5 px,event_recall 6-25 px,event_recall 26-100 px,event_recall >100 px
0,1,0.966,0.074,0.982,0.033,0.939,1.000,1.0,1.0
1,2,0.696,0.004,0.995,0.005,0.572,1.000,1.0,1.0
2,3,0.506,0.000,0.997,0.002,0.350,0.998,1.0,1.0


,mean,p50,p95,p99
gate1_ms,1.77,1.74,3.58,4.50
preprocess_ms,0.33,0.31,0.47,0.58
infer_ms,57.49,57.04,61.01,69.81
decide_ms,5.93,4.15,15.40,27.58
gate2_ms,0.15,0.15,0.19,0.23
encode_ms,0.05,0.05,0.08,0.10
total_ms,65.68,64.38,77.68,91.30


Host CPU latency only. Real CM4 latency/energy must come from CELL 19 benchmark bundle.
1239 packets queued for the next stage, 94.6 KB total.
Example decoded alert packet:
{
  "magic": 241,
  "version": 3,
  "model_id": 1,
  "frame_id": 0,
  "acquisition_timestamp_ms": 0,
  "n_events": 1,
  "flags": 64,
  "events": [
    {
      "lat_e7": 22284699,
      "lon_e7": 417499648,
      "x": 245,
      "y": 156,
      "w": 1,
      "h": 1,
      "area_ha_x10": 1,
      "confidence_u16": 64782,
      "metric_u16": 2089,
      "gsd_m_u16": 30,
      "geo_uncertainty_m_u16": 65535,
      "flags": 6
    }
  ]
}


In [20]:
# ================================================================
# CELL 18 — OPTIONAL: PERSISTENT PRIORITY QUEUE + CONTACT WINDOWS WITH COMMUNICATION LOSS
# ================================================================
# SQLite on the satellite's storage = survives a reboot.  Policy under test (A):
#   priority order (urgency desc) + persistent + failed/unsent packets stay queued and are retried.
# Compared with: B = FIFO order, C = priority but no persistence/retry (leftovers dropped each pass).
# Loss model per pass: lost contact (nothing sent), link cut mid-pass (packet in flight NOT acknowledged), or normal.
# This is optional integration logic for a later communications study; it is not part of the core
# raw-frame -> information-product objective and is not an orbit/link simulator.

class SqlQueue:
    def __init__(self, path, order='priority'):
        self.order = order
        self.con = sqlite3.connect(str(path))
        self.con.execute('PRAGMA journal_mode=WAL')
        self.con.execute('CREATE TABLE IF NOT EXISTS alerts (id INTEGER PRIMARY KEY, created_ts REAL, '
                         'urgency REAL, nbytes INTEGER, payload BLOB, state TEXT, attempts INTEGER DEFAULT 0, '
                         'sent_ts REAL)')
        self.con.execute("UPDATE alerts SET state='queued' WHERE state='sending'")   # crash recovery
        self.con.commit()

    def push(self, aid, created_ts, urgency, payload):
        self.con.execute("INSERT OR IGNORE INTO alerts (id, created_ts, urgency, nbytes, payload, state) "
                         "VALUES (?,?,?,?,?, 'queued')", (aid, created_ts, urgency, len(payload), payload))
        self.con.commit()

    def candidates(self):
        order = 'urgency DESC, created_ts ASC' if self.order == 'priority' else 'created_ts ASC'
        return self.con.execute(f"SELECT id, nbytes FROM alerts WHERE state='queued' ORDER BY {order}").fetchall()

    def mark_sent(self, aid, ts):
        self.con.execute("UPDATE alerts SET state='sent', sent_ts=?, attempts=attempts+1 WHERE id=?", (ts, aid))

    def mark_failed(self, aid, retry):
        self.con.execute("UPDATE alerts SET state=?, attempts=attempts+1 WHERE id=?",
                         ('queued' if retry else 'dropped', aid))

    def drop_remaining(self):
        self.con.execute("UPDATE alerts SET state='dropped' WHERE state='queued'")

    def commit(self):
        self.con.commit()

    def close(self):
        self.con.commit()
        self.con.close()

    def table(self):
        return pd.read_sql_query('SELECT id, created_ts, urgency, nbytes, state, attempts, sent_ts FROM alerts', self.con)


def draw_pass_outcomes(n_passes, p_lost, p_cut, seed):
    rng = np.random.default_rng(seed)
    out = []
    for _ in range(n_passes):
        u = rng.random()
        out.append(('lost', 0.0) if u < p_lost else ('cut', float(rng.uniform(0.2, 0.9))) if u < p_lost + p_cut else ('ok', 1.0))
    return out


def simulate(alerts, pass_times, pass_budget, outcomes, policy, db_path=':memory:', restart=False):
    """alerts: list of dict(id, created_ts, urgency, payload). Returns the final alert table."""
    q = SqlQueue(db_path, order=policy['order'])
    pending = sorted(alerts, key=lambda a: a['created_ts'])
    ptr = 0
    for t_pass, (outcome, cut_frac) in zip(pass_times, outcomes):
        while ptr < len(pending) and pending[ptr]['created_ts'] <= t_pass:
            a = pending[ptr]
            q.push(a['id'], a['created_ts'], a['urgency'], a['payload'])
            ptr += 1
        sent_bytes = 0
        limit = 0 if outcome == 'lost' else (pass_budget * cut_frac if outcome == 'cut' else pass_budget)
        for aid, nbytes in q.candidates():
            if sent_bytes + nbytes > pass_budget:
                break
            if sent_bytes + nbytes > limit:                       # link lost while this packet is in flight
                if outcome != 'lost':
                    q.mark_failed(aid, retry=policy['retry'])
                break
            sent_bytes += nbytes
            q.mark_sent(aid, t_pass)
        if not policy['persist']:
            q.drop_remaining()
        q.commit()
        if restart:                                               # simulate an onboard reboot between passes
            q.close()
            q = SqlQueue(db_path, order=policy['order'])
    table = q.table()
    q.close()
    return table


def summarize(table, alerts):
    urg = np.array([a['urgency'] for a in alerts])
    top = set(a['id'] for a in alerts if a['urgency'] >= np.quantile(urg, 0.8))
    sent = table[table['state'] == 'sent']
    delay = (sent['sent_ts'] - sent['created_ts']) / 3600.0
    top_sent = sent[sent['id'].isin(top)]
    top_delay = (top_sent['sent_ts'] - top_sent['created_ts']) / 3600.0
    return {'delivered_all': len(sent) / max(len(alerts), 1),
            'delivered_top20pct_urgency': len(top_sent) / max(len(top), 1),
            'mean_delay_h_top20pct': float(top_delay.mean()) if len(top_sent) else np.nan,
            'lost_forever': int((table['state'] == 'dropped').sum()),
            'retries': int((table['attempts'] - 1).clip(lower=0).sum())}


HORIZON_H, N_PASSES = 48.0, 12
P_LOST, P_CUT = 0.25, 0.25
POLICIES = {
    'A priority + persistent + retry': {'order': 'priority', 'persist': True, 'retry': True},
    'B FIFO + persistent + retry': {'order': 'fifo', 'persist': True, 'retry': True},
    'C priority, no persistence/retry': {'order': 'priority', 'persist': False, 'retry': False},
}

n_alerts = len(ALERTS)
assert n_alerts > 0, 'No alerts to queue (run CELL 17 first).'
alerts_sim = [{'id': k, 'created_ts': (k + 0.5) / n_alerts * HORIZON_H * 3600.0,
               'urgency': a['urgency'], 'payload': a['payload']} for k, a in enumerate(ALERTS)]
total_bytes = sum(len(a['payload']) for a in alerts_sim)
pass_times = [(j + 1) / N_PASSES * HORIZON_H * 3600.0 for j in range(N_PASSES)]

rows = []
for regime, capacity in (('stressed (50% of needed capacity)', 0.5), ('ample (150%)', 1.5)):
    budget = int(capacity * total_bytes / N_PASSES)
    for pname, pol in POLICIES.items():
        runs = [summarize(simulate(alerts_sim, pass_times, budget, draw_pass_outcomes(N_PASSES, P_LOST, P_CUT, s), pol),
                          alerts_sim) for s in range(30)]
        df_r = pd.DataFrame(runs)
        rows.append({'regime': regime, 'policy': pname, **{c: df_r[c].mean() for c in df_r.columns}})
queue_results_df = pd.DataFrame(rows)
display(queue_results_df.round(3))
queue_results_df.to_csv(OUTPUT_DIR / 'queue_policy_comparison_v7.csv', index=False)

# ---- persistence proof: reboot between passes must not change the outcome ----
outcomes0 = draw_pass_outcomes(N_PASSES, P_LOST, P_CUT, 123)
budget0 = int(0.5 * total_bytes / N_PASSES)
db_a, db_b = OUTPUT_DIR / 'queue_demo_norestart.db', OUTPUT_DIR / 'queue_demo_restart.db'
for p in (db_a, db_b, Path(str(db_a) + '-wal'), Path(str(db_b) + '-wal'), Path(str(db_a) + '-shm'), Path(str(db_b) + '-shm')):
    if p.exists():
        p.unlink()
t_a = simulate(alerts_sim, pass_times, budget0, outcomes0, POLICIES['A priority + persistent + retry'], db_a, restart=False)
t_b = simulate(alerts_sim, pass_times, budget0, outcomes0, POLICIES['A priority + persistent + retry'], db_b, restart=True)
same = (t_a.sort_values('id')[['id', 'state', 'attempts']].reset_index(drop=True)
        .equals(t_b.sort_values('id')[['id', 'state', 'attempts']].reset_index(drop=True)))
print('Reboot between every pass leaves the delivery outcome identical:', same)
assert same
print('Pass outcomes in this demo:', [o for o, _ in outcomes0])
print('Alerts:', n_alerts, '| total bytes:', total_bytes, '| bytes per pass (stressed):', budget0)


,regime,policy,delivered_all,delivered_top20pct_urgency,mean_delay_h_top20pct,lost_forever,retries
0,stressed (50% of needed capacity),A priority + persistent + retry,0.235,0.876,4.959,0.000,2.167
1,stressed (50% of needed capacity),B FIFO + persistent + retry,0.316,0.336,17.869,0.000,2.633
2,stressed (50% of needed capacity),"C priority, no persistence/retry",0.265,0.666,1.930,910.467,0.000
3,ample (150%),A priority + persistent + retry,0.821,0.994,3.210,0.000,2.200
4,ample (150%),B FIFO + persistent + retry,0.853,0.872,6.873,0.000,2.300
5,ample (150%),"C priority, no persistence/retry",0.690,0.755,1.930,384.400,0.000


Reboot between every pass leaves the delivery outcome identical: True
Pass outcomes in this demo: ['ok', 'lost', 'lost', 'lost', 'lost', 'ok', 'ok', 'cut', 'ok', 'ok', 'lost', 'ok']
Alerts: 1239 | total bytes: 94599 | bytes per pass (stressed): 3941


In [21]:
# ================================================================
# CELL 19 — COMPUTE BUDGET + HOST BENCHMARK + RASPBERRY-PI BENCHMARK BUNDLE
# ================================================================
# Host numbers are NOT onboard numbers. The bundle is for an on-target ARM benchmark; final flight
# acceptance requires measurement on the selected CubeSat processor (the cited 6U study used an RPi CM4).

BENCH_RUNS, BENCH_WARMUP = 50, 10
bench_frames = [preprocess_dn(np.asarray(BANDS_MM[int(i)][slots]), PRIMARY_NAMES)[None] for i in pipe_rows[:8]]


def bench(path, threads):
    so_ = ort.SessionOptions()
    so_.intra_op_num_threads = threads
    so_.inter_op_num_threads = 1
    s = ort.InferenceSession(str(path), so_, providers=['CPUExecutionProvider'])
    nm = s.get_inputs()[0].name
    for _ in range(BENCH_WARMUP):
        s.run(None, {nm: bench_frames[0]})
    ts = []
    for k in range(BENCH_RUNS):
        t0 = time.perf_counter()
        s.run(None, {nm: bench_frames[k % len(bench_frames)]})
        ts.append((time.perf_counter() - t0) * 1e3)
    ts = np.array(ts)
    return float(ts.mean()), float(np.percentile(ts, 50)), float(np.percentile(ts, 95))


cpu_name = platform.processor() or platform.machine()
try:
    for line in open('/proc/cpuinfo'):
        if line.startswith('model name'):
            cpu_name = line.split(':', 1)[1].strip()
            break
except OSError:
    pass

models_to_bench = {'FP32': onnx_fp32}
if onnx_int8 is not None:
    models_to_bench['INT8'] = onnx_int8
rows = []
macs = count_macs(primary_model, len(PRIMARY_NAMES))
for label, path in models_to_bench.items():
    for threads in (1, 4):
        mean_ms, p50, p95 = bench(path, threads)
        rows.append({'model': label, 'threads': threads, 'mean_ms': mean_ms, 'p50_ms': p50, 'p95_ms': p95,
                     'file_MB': size_mb(path), 'params': count_params(primary_model), 'GMACs': macs / 1e9,
                     'host_cpu': cpu_name, 'is_onboard_measurement': False})
benchmark_df = pd.DataFrame(rows)
display(benchmark_df.round(2))
benchmark_df.to_csv(OUTPUT_DIR / 'benchmark_host_v7.csv', index=False)
print('Host CPU:', cpu_name, '| these are NOT Raspberry Pi numbers.')

# ---------------- Raspberry Pi bundle ----------------
BENCH_PI_SOURCE = r'''#!/usr/bin/env python3
"""Benchmark the fire-detection U-Net on ANY machine (Raspberry Pi, ARM VM, laptop).
Usage:  python3 bench_pi.py --model model_int8.onnx --threads 1
Power:  python3 bench_pi.py --model model_int8.onnx --threads 1 --loop-seconds 120
        (read the USB power meter during the loop and during idle; see README.md)"""
import argparse, json, platform, subprocess, time
import numpy as np
import onnxruntime as ort

p = argparse.ArgumentParser()
p.add_argument('--model', default='model_int8.onnx')
p.add_argument('--threads', type=int, default=1)
p.add_argument('--runs', type=int, default=100)
p.add_argument('--warmup', type=int, default=10)
p.add_argument('--loop-seconds', type=float, default=0.0)
a = p.parse_args()


def cpu_model():
    try:
        for line in open('/proc/cpuinfo'):
            if line.startswith(('Model', 'model name', 'Hardware')):
                return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()


def sh(cmd):
    try:
        return subprocess.check_output(cmd, shell=True, text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return 'n/a'


x = np.load('sample_input.npy').astype(np.float32)
so = ort.SessionOptions()
so.intra_op_num_threads = a.threads
so.inter_op_num_threads = 1
sess = ort.InferenceSession(a.model, so, providers=['CPUExecutionProvider'])
name = sess.get_inputs()[0].name
for _ in range(a.warmup):
    sess.run(None, {name: x})
times = []
for _ in range(a.runs):
    t0 = time.perf_counter()
    sess.run(None, {name: x})
    times.append((time.perf_counter() - t0) * 1e3)
t = np.array(times)
report = {
    'cpu': cpu_model(), 'machine': platform.machine(), 'onnxruntime': ort.__version__,
    'model': a.model, 'threads': a.threads, 'input_shape': list(x.shape),
    'latency_ms_mean': round(float(t.mean()), 2), 'p50': round(float(np.percentile(t, 50)), 2),
    'p95': round(float(np.percentile(t, 95)), 2), 'p99': round(float(np.percentile(t, 99)), 2),
    'cpu_governor': sh('cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_governor'),
    'cpu_freq_mhz': sh('cat /sys/devices/system/cpu/cpu0/cpufreq/scaling_cur_freq'),
    'temp': sh('vcgencmd measure_temp'), 'throttled': sh('vcgencmd get_throttled'),
}
print(json.dumps(report, indent=2))

if a.loop_seconds > 0:
    print(f'Looping for {a.loop_seconds:.0f} s: note the power meter reading now ...')
    n, t_start = 0, time.perf_counter()
    while time.perf_counter() - t_start < a.loop_seconds:
        sess.run(None, {name: x})
        n += 1
    elapsed = time.perf_counter() - t_start
    print(json.dumps({'loop_inferences': n, 'loop_seconds': round(elapsed, 1),
                      'seconds_per_inference': round(elapsed / n, 4)}, indent=2))
    print('energy per inference (Wh) = (P_load_W - P_idle_W) * seconds_per_inference / 3600')
'''

BUNDLE = OUTPUT_DIR / 'pi_bundle'
BUNDLE.mkdir(exist_ok=True)
shutil.copy(onnx_fp32, BUNDLE / 'model_fp32.onnx')
if onnx_int8 is not None:
    shutil.copy(onnx_int8, BUNDLE / 'model_int8.onnx')
np.save(BUNDLE / 'sample_input.npy', bench_frames[0].astype(np.float32))
(BUNDLE / 'bench_pi.py').write_text(BENCH_PI_SOURCE)
(BUNDLE / 'README.md').write_text(f"""# Raspberry Pi benchmark bundle (primary model: {PRIMARY_CONFIG}, bands {PRIMARY_NAMES})

Target: Raspberry Pi 4 Model B / Compute Module 4 (Cortex-A72, the CPU class used by Chatar et al.);
KITSUNE itself used a CM3+ (Cortex-A53), so also run it there if you can get one.

## Run
```
sudo apt update && sudo apt install -y python3-venv python3-pip
python3 -m venv venv && . venv/bin/activate
pip install onnxruntime numpy
python3 bench_pi.py --model model_fp32.onnx --threads 1
python3 bench_pi.py --model model_int8.onnx --threads 1
python3 bench_pi.py --model model_int8.onnx --threads 4
```
Record: CPU governor (use `performance` for the final numbers), temperature and `throttled` (must be 0x0).
Use a heatsink/fan so the CPU does not throttle.

## Energy per inference (USB power meter between supply and Pi)
1. Idle: note P_idle (W) after 2 minutes with nothing running.
2. `python3 bench_pi.py --model model_int8.onnx --threads 1 --loop-seconds 120` and note P_load (W) during the loop.
3. energy_per_inference_Wh = (P_load - P_idle) * seconds_per_inference / 3600.
Compare with KITSUNE's reported 0.10 Wh per inference (MiniVGGNet on a CM3+, classification only).

## Emulation (no board)
`docker run --platform linux/arm64 ...` with QEMU only checks that the ARM build runs; its timings are not valid.

Input shape: {list(bench_frames[0].shape)} float32 (already preprocessed). Model MACs/frame: {macs / 1e9:.2f} G.
""")
print('Bundle written to', BUNDLE, '->', sorted(p.name for p in BUNDLE.iterdir()))


,model,threads,mean_ms,p50_ms,p95_ms,file_MB,params,GMACs,host_cpu,is_onboard_measurement
0,FP32,1,46.08,45.69,49.07,1.85,482737,2.32,Intel(R) Xeon(R) CPU @ 2.00GHz,False
1,FP32,4,24.19,24.01,25.86,1.85,482737,2.32,Intel(R) Xeon(R) CPU @ 2.00GHz,False
2,INT8,1,55.21,54.65,58.80,0.50,482737,2.32,Intel(R) Xeon(R) CPU @ 2.00GHz,False
3,INT8,4,32.17,31.60,36.16,0.50,482737,2.32,Intel(R) Xeon(R) CPU @ 2.00GHz,False


Host CPU: Intel(R) Xeon(R) CPU @ 2.00GHz | these are NOT Raspberry Pi numbers.
Bundle written to /kaggle/working/v7_outputs/pi_bundle -> ['README.md', 'bench_pi.py', 'model_fp32.onnx', 'model_int8.onnx', 'sample_input.npy']


In [ ]:
# ================================================================
# CELL 20 — NON-CIRCULAR CHECK: HUMAN-ANNOTATED SCENES (zero-shot, no fine-tuning)
# ================================================================
# Training labels are SWIR-algorithm outputs, so the human labels remove that circularity.
# Threshold and scalar temperature are fixed from the AFRICA validation set; no tuning on manual data.
# The result is a non-circular check, not a flight validation.
RUN_DELETE_MANUAL_AFTER = True
manual_results_df = None


def clean_manual_stem(stem):
    stem = re.sub(r'_(?:manual|annotation|annotations|mask)$', '', str(stem), flags=re.IGNORECASE)
    return re.sub(r'_v1(?=_p\d+$)', '', stem, flags=re.IGNORECASE)


def _tifs(root):
    return sorted(p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in ('.tif', '.tiff'))


def discover_manual_files(root):
    imgs, anns = _tifs(Path(root) / 'landsat_images'), _tifs(Path(root) / 'manual_annotations')
    sub_i = [p for p in imgs if any(part.lower() == 'images' for part in p.parts)]
    sub_a = [p for p in anns if any(part.lower() in {'annotations', 'annotation'} for part in p.parts)]
    return (sub_i or imgs), (sub_a or anns)


def run_manual_zero_shot():
    import gdown
    root = SCRATCH_ROOT / 'manual_release'
    root.mkdir(parents=True, exist_ok=True)
    ids = {'landsat_images': '1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg',
           'manual_annotations': '1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf'}
    for name, fid in ids.items():
        target, marker = root / name, root / name / '.extracted_ok'
        if marker.exists():
            continue
        archive = root / f'{name}.zip'
        if not (archive.exists() and zipfile.is_zipfile(archive)):
            archive.unlink(missing_ok=True)
            print('Downloading', name, '...')
            gdown.download(id=fid, output=str(archive), quiet=False, use_cookies=False)
        assert archive.exists() and zipfile.is_zipfile(archive), f'{name}: invalid ZIP'
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(target)
        marker.write_text('ok')
        archive.unlink()
        print('Extracted', name)

    images, annotations = discover_manual_files(root)
    print('images:', len(images), '| annotations:', len(annotations))
    by_stem = {clean_manual_stem(p.stem): p for p in images}
    pairs = [(by_stem[clean_manual_stem(a.stem)], a) for a in annotations if clean_manual_stem(a.stem) in by_stem]
    rate = len(pairs) / max(len(annotations), 1)
    print(f'paired: {len(pairs)} ({rate:.1%})')
    if rate < 0.95:
        print('Unpaired examples:', [a.stem for a in annotations[:3]], list(by_stem)[:3])
        raise RuntimeError('Pairing rate < 95%: fix filename normalization first.')
    scene_of = lambda p: re.sub(r'_p\d+$', '', clean_manual_stem(p.stem))

    models = {}
    for cfg, info in TRAINED.items():
        m, _ = load_model_from_checkpoint(info['ckpt'])
        models[cfg] = (m, channel_slots(info['names']), info['names'],
                       OPERATING[cfg]['threshold'], OPERATING[cfg]['temperature'])
    acc = {cfg: dict(tp=0, fp=0, fn=0, fire=0, hit=0, nofire=0, fa=0, ev=0, found=0, pe=0, fe=0) for cfg in models}
    scene_alarm = {cfg: defaultdict(lambda: [0, 0, 0]) for cfg in models}

    def load(pair):
        img, ann = pair
        arr, _ = read_patch_bands(img, CACHE_BAND_NUMBERS)
        with rasterio.open(ann) as src:
            msk = _pad_to((src.read(1) > 0).astype(np.uint8))[0]
        return arr, msk, scene_of(img)

    CH = 32
    with ThreadPoolExecutor(max_workers=CACHE_THREADS) as ex:
        for s in tqdm(range(0, len(pairs), CH), desc='Manual scenes'):
            loaded = list(ex.map(load, pairs[s:s + CH]))
            gts = [(m > 0) for _, m, _ in loaded]
            for cfg, (model, sl, names, thr, temp) in models.items():
                x = np.stack([preprocess_dn(arr[sl], names) for arr, _, _ in loaded])
                with torch.no_grad():
                    logits = model(torch.from_numpy(x).to(device)).cpu().numpy()
                prob = sigmoid_temperature_np(logits, temp)[:, 0]
                a = acc[cfg]
                for p, g, (_, _, sc) in zip(prob, gts, loaded):
                    pred = p >= thr
                    a['tp'] += int((pred & g).sum()); a['fp'] += int((pred & ~g).sum()); a['fn'] += int((~pred & g).sum())
                    if g.any():
                        a['fire'] += 1; a['hit'] += int(pred.any())
                    else:
                        a['nofire'] += 1; a['fa'] += int(pred.any())
                    sizes, found, npd, nfa = match_events(pred, g)
                    a['ev'] += len(sizes); a['found'] += int(found.sum()); a['pe'] += npd; a['fe'] += nfa
                    sa = scene_alarm[cfg][sc]
                    sa[0] += 1; sa[1] += int(g.any()); sa[2] += int(pred.any())

    rows = []
    for cfg, a in acc.items():
        m = metrics_from_counts(a['tp'], a['fp'], a['fn'])
        rows.append({'config': cfg, 'bands': '+'.join(TRAINED[cfg]['names']),
                     'temperature': models[cfg][4], 'benchmark_threshold': models[cfg][3],
                     'precision': m['precision'], 'recall': m['recall'], 'IoU': m['iou'], 'F1': m['f1'],
                     'frame_recall': a['hit'] / max(a['fire'], 1), 'frame_FAR': a['fa'] / max(a['nofire'], 1),
                     'event_recall': a['found'] / max(a['ev'], 1),
                     'event_precision': 1 - a['fe'] / max(a['pe'], 1)})
    result = pd.DataFrame(rows)
    display(result.round(3))
    print('This is the independent human-label check. It does NOT make the model flight-ready.')
    sc_rows = [{'config': cfg, 'scene': sc, 'frames': v[0], 'frames_with_fire': v[1], 'frames_with_alarm': v[2]}
               for cfg, d in scene_alarm.items() for sc, v in d.items()]
    scene_table = pd.DataFrame(sc_rows)
    scene_table.to_csv(OUTPUT_DIR / 'manual_scene_alarms_v7.csv', index=False)
    print('\nFire-free scenes (every alarm here is a false alarm):')
    display(scene_table[scene_table['frames_with_fire'] == 0].round(3))
    result.to_csv(OUTPUT_DIR / 'manual_zero_shot_v7.csv', index=False)
    if RUN_DELETE_MANUAL_AFTER:
        shutil.rmtree(root, ignore_errors=True)
    return result


if RUN_MANUAL_ZERO_SHOT:
    try:
        manual_results_df = run_manual_zero_shot()
    except Exception as exc:
        print('⚠️ Manual zero-shot evaluation skipped:', repr(exc)[:400])
else:
    print('RUN_MANUAL_ZERO_SHOT = False -> skipped.')

Downloading...
From (original): https://drive.google.com/uc?id=1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg
From (redirected): https://drive.google.com/uc?id=1uZnc65_GRFdAoavGoUKkJfeVQOUI8lVg&confirm=t&uuid=367a1a8d-b34e-4bbc-9f50-3a2c79dfd045
To: /kaggle/tmp/fireedge/manual_release/landsat_images.zip
100%|██████████| 9.61G/9.61G [02:34<00:00, 62.3MB/s]


Extracted landsat_images


Downloading...
From: https://drive.google.com/uc?id=1LdsX-rH5hy_82jfc1akO8p4n0_N8lRgf
To: /kaggle/tmp/fireedge/manual_release/manual_annotations.zip
100%|██████████| 82.9k/82.9k [00:00<00:00, 45.8MB/s]


Extracted manual_annotations
images: 9044 | annotations: 100
paired: 100 (100.0%)


Manual scenes:   0%|          | 0/4 [00:00<?, ?it/s]

,config,bands,temperature,benchmark_threshold,precision,recall,IoU,F1,frame_recall,frame_FAR,event_recall,event_precision
0,pereira3c,c7+c6+c2,2.123,0.96,0.762,0.959,0.738,0.849,0.99,0.0,0.929,0.801
1,b10,b10,1.969,0.29,0.715,0.276,0.249,0.399,0.43,0.0,0.328,0.920
2,pereira3c_b10,c7+c6+c2+b10,2.177,0.60,0.839,0.948,0.801,0.890,0.99,0.0,0.911,0.818


This is the independent human-label check. It does NOT make the model flight-ready.

Fire-free scenes (every alarm here is a false alarm):


,config,scene,frames,frames_with_fire,frames_with_alarm


In [23]:
# ================================================================
# CELL 21 — FINAL TABLES + DEPLOYMENT MANIFEST + RESULTS ZIP
# ================================================================
def _records(df):
    return None if df is None else json.loads(df.to_json(orient='records'))


primary_operating = OPERATING[PRIMARY_CONFIG]

manifest = {
    'version': 'V7 (multiband information-product pipeline)',
    'bands_cached': CACHE_BAND_NAMES,
    'configurations': {k: BAND_CONFIGS[k] for k in TRAINED},
    'reference_configuration': REFERENCE_CONFIG,
    'payload_proxy_configuration': FLIGHT_CONFIG,
    'thermal_control_configuration': CONTROL_CONFIG,
    'primary_configuration': PRIMARY_CONFIG,
    'primary_bands': PRIMARY_NAMES,
    'primary_role': 'payload_proxy_flight_candidate',
    'model_ids': MODEL_IDS,
    'label_source': 'Pereira voting mask (>=2 of 3 reflectance-based algorithms; no thermal band involved)',
    'split': 'scene-level 70/15/15, seed 42',
    'model': {
        'architecture': 'Tiny U-Net, standard 3x3 convs',
        'base': MODEL_BASE,
        'params': count_params(primary_model),
        'GMACs_per_frame': count_macs(primary_model, len(PRIMARY_NAMES)) / 1e9,
    },
    'probability_calibration': {
        'method': 'scalar logit temperature scaling on AFRICA validation pixels',
        'temperature': PRIMARY_TEMPERATURE,
        'calibration_pixels': primary_operating['calibration_pixels'],
    },
    'threshold': {
        'benchmark_threshold': PRIMARY_THRESHOLD,
        'selection_rule': primary_operating['rule'],
        'constraints_met_on_validation': bool(primary_operating['constraints_met']),
        'min_frame_recall_target': MIN_FRAME_RECALL,
        'max_frame_far_target': MAX_FRAME_FAR,
        'frame_recall_gap': primary_operating['frame_recall_gap_to_target'],
        'far_margin': primary_operating['far_margin_to_limit'],
        'note': 'When constraints are infeasible, the threshold is a max-F1 benchmark point, not a demonstrated mission operating point.',
    },
    'packet': {
        'version': PKT_VERSION,
        'header_bytes': HEADER_BYTES,
        'event_bytes': EVENT_BYTES,
        'max_events': MAX_EVENTS,
        'max_alert_packet_bytes': MAX_ALERT_PACKET_BYTES,
        'max_empty_or_heartbeat_bytes': MAX_EMPTY_PACKET_BYTES,
        'fields': [
            'acquisition_timestamp_ms', 'model_id', 'event latitude/longitude',
            'pixel bbox', 'area_ha_x10 (0.1 ha units)', 'mean calibrated confidence',
            'event contrast metric', 'GSD_m', 'geolocation uncertainty', 'flags', 'CRC32'
        ],
        'metric_semantics': 'B10 = thermal hotspot contrast in K*100; Pereira-3C = SWIR2 local spectral contrast in reflectance*10000.',
    },
    'heartbeat': {
        'every_n_frames': HEARTBEAT_EVERY_N_FRAMES,
        'purpose': 'distinguish no-fire silence from an offline / failed processor.',
    },
    'pipeline_model': globals().get('PIPELINE_MODEL'),
    'pipeline': globals().get('pipeline_summary'),
    'test_results_by_config': _records(results_df),
    'robustness': _records(globals().get('robust_df')),
    'onnx_verification': _records(globals().get('verification_df')),
    'int8_within_tolerance': bool(globals().get('INT8_OK', False)),
    'queue_policies': _records(globals().get('queue_results_df')),
    'host_benchmark': _records(globals().get('benchmark_df')),
    'manual_zero_shot': _records(globals().get('manual_results_df')),
    'artifacts': {
        'fp32_onnx': str(onnx_fp32),
        'int8_onnx': str(onnx_int8) if onnx_int8 else None,
        'sample_alert_packet_bin': str(OUTPUT_DIR / 'sample_alert_packet.bin') if (OUTPUT_DIR / 'sample_alert_packet.bin').exists() else None,
        'sample_alert_packet_json': str(OUTPUT_DIR / 'sample_alert_packet.json') if (OUTPUT_DIR / 'sample_alert_packet.json').exists() else None,
        'pi_bundle': str(OUTPUT_DIR / 'pi_bundle'),
    },
    'limitations': [
        'Pereira-3C (C7+C6+C2) is the current SWIR3 payload proxy; it is not the final flight sensor calibration.',
        'B10 is a thermal control experiment only; it is not the current payload choice. Landsat B10 is also only a sensor proxy.',
        'Landsat is not a microbolometer: no true NUC stage or sensor-specific microbolometer artifact model is implemented.',
        'Training labels are SWIR-algorithm outputs; SWIR inputs therefore have a built-in advantage on the voting-mask test.',
        'The dataset is detection-curated and fire-heavy; it is not an orbit-like prevalence test set.',
        'The validation frame-recall/FAR mission targets may be infeasible; V7 reports that explicitly instead of silently claiming success.',
        'Gate 1 scrubbing is disabled by default because isolated 1-pixel fire detections can be removed; the robustness cell measures the trade-off.',
        'Acquisition timestamps and absolute geolocation uncertainty are not available in the current Landsat patch interface; packets flag these as unavailable rather than inventing values.',
        'The current pipeline GSD is a 30 m Landsat reference value; flight GSD must be supplied by the payload/geometry layer.',
        'Latency and energy are host/logic outputs until the Raspberry Pi CM4/target board benchmark is physically measured.',
        'Queue/retry is a logic prototype, not an orbit/link simulator.',
    ],
}
with open(OUTPUT_DIR / 'manifest_v7.json', 'w') as f:
    json.dump(manifest, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

print('========== TEST RESULTS BY BAND CONFIGURATION ==========')
cols = ['config', 'bands', 'params', 'MACs_G', 'temperature', 'threshold', 'constraints_met', 'F1', 'recall', 'frame_recall', 'false_alarm_rate']
display(results_df[[c for c in cols if c in results_df.columns] +
                    [c for c in results_df.columns if c.startswith('event_recall')]].round(3))
if manual_results_df is not None:
    print('\n========== HUMAN-ANNOTATED SCENES (zero-shot) ==========')
    display(manual_results_df.round(3))

archive = shutil.make_archive(str(WORK_ROOT / 'v7_results'), 'zip', OUTPUT_DIR)
print('\nResults archive:', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
print('Artifacts written to:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print('  ', p.name)

========== TEST RESULTS BY BAND CONFIGURATION ==========


,config,bands,params,MACs_G,temperature,threshold,constraints_met,F1,recall,frame_recall,false_alarm_rate,event_recall_all,event_recall 1-5 px,event_recall 6-25 px,event_recall 26-100 px,event_recall >100 px
0,pereira3c,c7+c6+c2,482737,2.319,2.123,0.96,True,0.949,0.949,0.965,0.063,0.955,0.946,1.000,1.000,1.0
1,b10,b10,482449,2.301,1.969,0.29,False,0.412,0.328,0.180,0.026,0.133,0.053,0.455,0.823,1.0
2,pereira3c_b10,c7+c6+c2+b10,482881,2.329,2.177,0.60,True,0.946,0.926,0.964,0.067,0.945,0.934,1.000,1.000,1.0



========== HUMAN-ANNOTATED SCENES (zero-shot) ==========


,config,bands,temperature,benchmark_threshold,precision,recall,IoU,F1,frame_recall,frame_FAR,event_recall,event_precision
0,pereira3c,c7+c6+c2,2.123,0.96,0.762,0.959,0.738,0.849,0.99,0.0,0.929,0.801
1,b10,b10,1.969,0.29,0.715,0.276,0.249,0.399,0.43,0.0,0.328,0.920
2,pereira3c_b10,c7+c6+c2+b10,2.177,0.60,0.839,0.948,0.801,0.890,0.99,0.0,0.911,0.818



Results archive: /kaggle/working/v7_results.zip (121.6 MB)
Artifacts written to: /kaggle/working/v7_outputs
   africa_inventory_split_v7.csv
   africa_inventory_split_v7_pre_cache.csv
   band_information_probe.csv
   benchmark_host_v7.csv
   epoch_checkpoints
   epoch_mission_selection_b10.csv
   epoch_mission_selection_pereira3c.csv
   epoch_mission_selection_pereira3c_b10.csv
   event_confidence_by_size_test_v7.csv
   event_confidence_calibration_v7.json
   event_reliability_test_v7.csv
   manifest_v7.json
   manual_scene_alarms_v7.csv
   manual_zero_shot_v7.csv
   onnx_verification_v7.csv
   pi_bundle
   pipeline_event_quality_v7.csv
   pipeline_latency_host_v7.csv
   probability_calibration_b10.json
   probability_calibration_pereira3c.json
   probability_calibration_pereira3c_b10.json
   queue_demo_norestart.db
   queue_demo_restart.db
   queue_policy_comparison_v7.csv
   robustness_primary_v7.csv
   sample_alert_packet.bin
   sample_alert_packet.json
   split_summary_v7.csv
   t

In [24]:
# ============================================================
# CELL 22 — CREATE FINAL V7 ZIP PACKAGE
# ============================================================

from pathlib import Path
from datetime import datetime
import zipfile
from IPython.display import FileLink, display

# ------------------------------------------------------------
# 1) Locate OUTPUT_DIR
# ------------------------------------------------------------
OUTPUT_DIR = Path(OUTPUT_DIR)

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(
        f"OUTPUT_DIR does not exist: {OUTPUT_DIR}"
    )

# ------------------------------------------------------------
# 2) Create timestamped ZIP
# ------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

zip_path = Path("/kaggle/working") / (
    f"fire_detection_V7_outputs_{timestamp}.zip"
)

# ------------------------------------------------------------
# 3) Package everything in OUTPUT_DIR
# ------------------------------------------------------------
file_count = 0

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6
) as zf:

    for path in OUTPUT_DIR.rglob("*"):

        if not path.is_file():
            continue

        # Don't accidentally include the ZIP itself
        if path.resolve() == zip_path.resolve():
            continue

        # Preserve OUTPUT_DIR structure
        arcname = path.relative_to(OUTPUT_DIR)

        zf.write(path, arcname=arcname)
        file_count += 1

# ------------------------------------------------------------
# 4) Report
# ------------------------------------------------------------
zip_size_mb = zip_path.stat().st_size / (1024 ** 2)

print("=" * 70)
print("V7 FINAL OUTPUT PACKAGE")
print("=" * 70)
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"ZIP        : {zip_path}")
print(f"Files      : {file_count}")
print(f"Size       : {zip_size_mb:.2f} MB")
print("=" * 70)

# ------------------------------------------------------------
# 5) Create a clickable download link
# ------------------------------------------------------------
display(FileLink(
    str(zip_path),
    result_html_prefix="DOWNLOAD ZIP: "
))

print("\nZIP READY.")

V7 FINAL OUTPUT PACKAGE
OUTPUT_DIR : /kaggle/working/v7_outputs
ZIP        : /kaggle/working/fire_detection_V7_outputs_20261008_202441.zip
Files      : 108
Size       : 115.94 MB


/kaggle/working/fire_detection_V7_outputs_20261008_202441.zip


ZIP READY.
